# AI Tối Ưu Lịch Trình MTA — 29 tuyến

Pipeline: dữ liệu → NN demand → tối ưu lịch trình.

**Setup:** `DATA_DIR`, `SCHEDULE_DIR`, `RUN_EXPERIMENT` (`default` | `model_mlp` | `model_lstm`). Kết quả: `OUT_DIR`.

**Ghi chú kết quả (full run `default`, blend MLP+HistGBM):** hold-out autumn_2025 — Model MAE **481** (−48.9% vs baseline); analytical opt weekday_peak — chờ **−24.5%**, objective **−21%** (λ_eval=150), trips **+23%**. Chi tiết §9–10.

**Ablation 1 tuyến:** `mta_schedule_optimization_single_route.ipynb`


## 1. Setup & Imports

In [1]:
import os
import json
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

import sys
import importlib

LIB_DIR = Path("/kaggle/input/datasets/tnguynthnh142/libmta")
DATA_DIR = Path("/kaggle/input/datasets/tnguynthnh142/mta-subway-ny")
SCHEDULE_DIR = Path("/kaggle/input/datasets/tnguynthnh142/schedule-subway")
if not DATA_DIR.exists():
    DATA_DIR = Path("../datasets")
    SCHEDULE_DIR = Path("../datasets/schedule_current")
    print("Fallback local:", DATA_DIR.resolve())
_REPO_ROOT = Path(".").resolve()
_REPO_LIB = _REPO_ROOT / "lib" / "single_route_pipeline.py"
if _REPO_LIB.exists():
    sys.path.insert(0, str(_REPO_ROOT))
    import lib.single_route_pipeline as srp
elif (LIB_DIR / "single_route_pipeline.py").exists():
    sys.path.insert(0, str(LIB_DIR))
    import single_route_pipeline as srp
else:
    raise FileNotFoundError(f"Không tìm thấy single_route_pipeline.py tại {_REPO_LIB} hoặc {LIB_DIR}")
importlib.reload(srp)


RUN_EXPERIMENT = "default"  # default | model_mlp | model_lstm
srp.apply_experiment(RUN_EXPERIMENT, globals())

OUT_DIR = Path("/kaggle/working") / RUN_EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = DATA_DIR / "train_manifest.json"
if MANIFEST_FILE.exists():
    with open(MANIFEST_FILE, encoding="utf-8") as f:
        train_manifest = json.load(f)
else:
    train_manifest = {}

_DEFAULT_PATHS = {
    "ridership": "ridership.csv",
    "routes_by_station_complex": "routes_by_station_complex.csv",
    "routes": "routes.csv",
    "factors_hourly": "factors_hourly.csv",
    "headway_by_route_hour": "headway_by_route_hour.csv",
    "factors_daily": "factors_daily.csv",
}

def data_path(key: str) -> Path:
    """Resolve path trong DATA_DIR (ridership bundle); không dùng cho GTFS."""
    paths_cfg = train_manifest.get("paths", {}) or {}
    name = paths_cfg.get(key, _DEFAULT_PATHS.get(key, key))
    return DATA_DIR / name

RIDERSHIP_FILE = data_path("ridership")
ROUTES_BY_COMPLEX_FILE = data_path("routes_by_station_complex")
FACTORS_HOURLY_FILE = data_path("factors_hourly")
HEADWAY_FILE = data_path("headway_by_route_hour")

FACTORS_MODE = "hourly"
FACTORS_JOIN_KEYS = ["date", "hour"]

_scope = train_manifest.get("scope", {})
OPT_HOURS_FALLBACK = _scope.get("hours", list(range(24)))
OPT_HOURS = list(OPT_HOURS_FALLBACK)
USE_GTFS_OPERATING_HOURS = True
MIN_GTFS_TRIPS_PER_HOUR = 1

OPT_ROUTE_MODE = _scope.get("route_mode", "headway")
DIRECTIONS = _scope.get("directions", [0, 1])
MIN_ROUTE_DEMAND_ROWS = 50
ROUTE_ALIASES = train_manifest.get("route_aliases", {"SIR": "SI"})
HEADWAY_SERVICE = "Weekday"

COVERAGE_THR = 0.7
CHUNK_ROWS = 1_000_000
NN_EPOCHS_MAIN = 200
NN_EPOCHS_CV = 80
CV_SPLITS = 8
CV_SEASON_YEARS = 2
HOLDOUT_TEST_SEASON = "autumn"
HOLDOUT_TEST_YEAR = 2025
HOLDOUT_VAL_SEASON = "summer"
HOLDOUT_VAL_YEAR = 2025

USE_ROUTE_EMBEDDING = True
USE_LAG_FEATURES = True
LAG_FEATURE_COLS = ["log_lag_24h", "log_lag_168h", "log_rolling_7d"]
RESID_CLIP = (-0.4, 0.4)
RESID_CLIP_CANDIDATES = [0.35, 0.45, 0.55]
CV_BLEND_TUNE_FRAC = 0.15
PEAK_SAMPLE_WEIGHT = 2.0
NN_HIDDEN = (64, 32)
NN_DROPOUT = 0.25
USE_BATCH_NORM = True
LAMBDA_COST = 150.0
LAMBDA_GRID = [100, 150, 200, 400, 600, 1000]

USE_DEMAND_SPILLOVER = True
SPILLOVER_ALPHA = 0.05
SPILLOVER_ALPHA_GRID = [0.0, 0.02, 0.05, 0.08, 0.10]
TRANSFER_MIN_WEIGHT = 0.1

TRIPS_MIN_FACTOR = 0.5
TRIPS_PEAK_MAX_FACTOR = 1.15
TRIPS_OFFPEAK_MAX_FACTOR = 1.35
TRIPS_OVERNIGHT_MAX_FACTOR = 1.10
TRIPS_MAX_DELTA = 3
LAMBDA_AUTO_CALIBRATE = True
USE_ANALYTICAL_OPT = True   # full 29 routes: analytical
RUN_GA_TABU = True          # §7–8: chạy khi USE_ANALYTICAL_OPT=False và N_SLOTS≤GA_MAX_SLOTS
GA_POP_SIZE = 80
GA_GENERATIONS = 120
TABU_ITERS = 400
GA_MAX_SLOTS = 200
TARGET_MAX_BIND_FRACTION = 0.50
SPLIT_DEMAND_BY_DIRECTION = True

OPT_TARGET = "balanced"
BALANCED_WEIGHTS = {"wait": 0.6, "cost": 0.4}
LAMBDA_CANDIDATES = [60.0, 80.0, 100.0, 120.0, 150.0, 180.0, 220.0, 260.0]


HOURLY_FACTOR_COLS = [
    "temperature_c", "apparent_temperature_c",
    "precipitation_mm", "rain_mm", "snowfall_cm",
    "windspeed_kmh", "windgusts_kmh",
    "is_rain", "is_snow", "is_severe_wind",
    "is_peak_morning", "is_peak_evening", "is_overnight",
    "is_major_event_window",
]


print("TensorFlow:", tf.__version__)
print(f"Run: {RUN_EXPERIMENT} → model={DEMAND_MODEL_TYPE} | out={OUT_DIR.resolve()}")
for label, p in [("lib", LIB_DIR / "single_route_pipeline.py"), ("ridership", RIDERSHIP_FILE),
                 ("factors_hourly", FACTORS_HOURLY_FILE), ("schedule", SCHEDULE_DIR)]:
    print(f"  {label}: {'OK' if p.exists() else 'MISSING'} — {p}")


Fallback local: D:\tranport-public\datasets
TensorFlow: 2.20.0
Run: default → model=blend | out=D:\kaggle\working\default
  lib: MISSING — \kaggle\input\datasets\tnguynthnh142\libmta\single_route_pipeline.py
  ridership: OK — ..\datasets\ridership.csv
  factors_hourly: OK — ..\datasets\factors_hourly.csv
  schedule: OK — ..\datasets\schedule_current


## 2. Load & Hợp nhất dữ liệu thực

Bộ dữ liệu trong [`datasets/`](../datasets/) — khoá join tại [`datasets/train_manifest.json`](../datasets/train_manifest.json):

- `ridership.csv` — nhu cầu theo giờ × ga (đọc theo chunk vì file lớn).
- `routes_by_station_complex.csv` — bảng map ga → tuyến GTFS.
- `routes.csv` — metadata tuyến GTFS.
- `schedule_current/{trips,stop_times,calendar,calendar_dates}.txt` — GTFS hiện tại; baseline headway được **tự dựng** từ đây (không còn file tổng hợp `headway_by_route_hour.csv`).
- `factors_hourly.csv` — thời tiết + lịch **theo giờ** (join `date` + `hour`).

**Merge**: ridership (chunked) → aggregate `(station_complex_id, date, hour)` → map sang route → join **factors_hourly** trên `(date, hour)`.

In [2]:
routes_complex = pd.read_csv(ROUTES_BY_COMPLEX_FILE)
routes_complex["station_complex_id"] = routes_complex["station_complex_id"].astype(str)

route_col = "gtfs_route_ids" if "gtfs_route_ids" in routes_complex.columns else "route_ids_gtfs"
station_to_routes = (
    routes_complex.assign(route=routes_complex[route_col].astype(str).str.split())
    .explode("route")
    .dropna(subset=["route"])
)
station_to_routes = station_to_routes[station_to_routes["route"] != ""]
station_to_routes = station_to_routes[["station_complex_id", "route"]].drop_duplicates()
station_to_routes["route"] = station_to_routes["route"].replace(ROUTE_ALIASES)

route_counts = station_to_routes.groupby("station_complex_id").size().rename("n_routes_at_station")
station_to_routes = station_to_routes.merge(route_counts, on="station_complex_id")
station_to_routes["weight"] = 1.0 / station_to_routes["n_routes_at_station"]

print("Tổng map station→route:", len(station_to_routes))
print("Số ga có map:", station_to_routes["station_complex_id"].nunique())
print("Số tuyến xuất hiện:", station_to_routes["route"].nunique())
station_to_routes.head()


Tổng map station→route: 767
Số ga có map: 445
Số tuyến xuất hiện: 26


,station_complex_id,route,n_routes_at_station,weight
0,1,N,2,0.5
1,1,W,2,0.5
2,2,N,2,0.5
3,2,W,2,0.5
4,3,N,2,0.5


In [ ]:
usecols = ["transit_timestamp", "transit_mode", "station_complex_id", "ridership"]
valid_stations = set(station_to_routes["station_complex_id"].unique())

agg_parts = []
for chunk in pd.read_csv(
    RIDERSHIP_FILE,
    usecols=usecols,
    chunksize=CHUNK_ROWS,
    parse_dates=["transit_timestamp"],
    dtype={"station_complex_id": str},
):
    chunk = chunk[chunk["transit_mode"] == "subway"]
    chunk = chunk[chunk["station_complex_id"].isin(valid_stations)]
    if chunk.empty:
        continue
    chunk["date"] = chunk["transit_timestamp"].dt.date
    chunk["hour"] = chunk["transit_timestamp"].dt.hour
    grp = (
        chunk.groupby(["station_complex_id", "date", "hour"], as_index=False)["ridership"]
        .sum()
    )
    agg_parts.append(grp)

ridership_station = pd.concat(agg_parts, ignore_index=True)
print(f"Aggregated (station,date,hour) rows: {len(ridership_station):,}")
print("Date range:", ridership_station["date"].min(), "→", ridership_station["date"].max())


In [ ]:
ridership_station = (
    ridership_station.groupby(["station_complex_id", "date", "hour"], as_index=False)["ridership"]
    .sum()
)

routed = ridership_station.merge(station_to_routes, on="station_complex_id", how="inner")
routed["ridership_route"] = routed["ridership"] * routed["weight"]

route_stations_total = station_to_routes.groupby("route").size().rename("n_stations_route")

agg = routed.groupby(["route", "date", "hour"]).agg(
    demand_observed=("ridership_route", "sum"),
    n_active=("station_complex_id", "nunique"),
).reset_index()
agg = agg.merge(route_stations_total, left_on="route", right_index=True)

agg["coverage"] = agg["n_active"] / agg["n_stations_route"]
agg = agg[agg["coverage"] >= COVERAGE_THR].copy()
agg["demand"] = agg["demand_observed"] / agg["coverage"]

ridership_route = agg[["route", "date", "hour", "demand", "coverage"]].rename(
    columns={"route": "route_id"}
)
ridership_route["date"] = pd.to_datetime(ridership_route["date"])


print(f"Total (route,date,hour) records (coverage≥{COVERAGE_THR}): {len(ridership_route):,}")
print(f"Coverage stats: mean={ridership_route['coverage'].mean():.2f}, "
      f"min={ridership_route['coverage'].min():.2f}, "
      f"max={ridership_route['coverage'].max():.2f}")
print(f"Unique dates: {ridership_route['date'].nunique()}")
ridership_route.head()

In [ ]:
def load_factors_hourly(path: Path) -> pd.DataFrame:
    """Load hourly weather/calendar; one row per (date, hour).
    Join key: ridership(date,hour) ↔ factors_hourly(date,hour) — chuẩn hoá từ
    SUBSTR(transit_timestamp,1,19) = timestamp (merge hint trong train_manifest.json).
    """
    fh = pd.read_csv(path, parse_dates=["timestamp", "date"])
    if "hour" not in fh.columns:
        fh["hour"] = fh["timestamp"].dt.hour
    keep = [
        "date", "hour", "day_of_week", "is_weekend", "is_us_holiday", "month",
        *HOURLY_FACTOR_COLS,
    ]
    keep = [c for c in keep if c in fh.columns]
    fh = fh[keep].drop_duplicates(subset=["date", "hour"]).copy()
    fh["date"] = pd.to_datetime(fh["date"]).dt.normalize()
    fh["hour"] = fh["hour"].astype(int)
    return fh


def _gtfs_time_to_minutes(t: str) -> float:
    """GTFS departure_time: HH:MM:SS, có thể >24h (qua nửa đêm)."""
    if not isinstance(t, str) or ":" not in t:
        return np.nan
    h, m, s = t.split(":")
    return int(h) * 60 + int(m) + int(s) / 60.0


def build_headway_from_gtfs(
    schedule_dir: Path,
    service_id: str = HEADWAY_SERVICE,
) -> pd.DataFrame:
    """Dựng baseline headway (route × direction × hour) từ schedule_current/*.txt.

    Quy ước:
      - Lấy giờ khởi hành đại diện = `departure_time` ở stop đầu tiên (stop_sequence nhỏ nhất).
      - Lọc trip theo `service_id` (default Weekday); nếu service đó không có dữ liệu thì fallback
        sang service đầu tiên có sẵn trong calendar.txt.
      - Giờ vượt 24 được mod 24 để khớp scope tối ưu 0..23.
      - Khi `direction_id` thiếu → mặc định 0.
    Trả về DataFrame: route_id, direction_id, hour, trip_count, avg_headway_min, min_headway_min.
    """
    schedule_dir = Path(schedule_dir)
    trips = pd.read_csv(
        schedule_dir / "trips.txt",
        dtype={"trip_id": str, "route_id": str, "service_id": str},
    )
    stop_times = pd.read_csv(
        schedule_dir / "stop_times.txt",
        dtype={"trip_id": str, "stop_id": str, "departure_time": str, "arrival_time": str},
        usecols=lambda c: c in {"trip_id", "stop_sequence", "departure_time", "arrival_time"},
    )

    cal_path = schedule_dir / "calendar.txt"
    available_services = trips["service_id"].dropna().unique().tolist()
    if cal_path.exists():
        cal = pd.read_csv(cal_path, dtype={"service_id": str})
        available_services = cal["service_id"].dropna().unique().tolist() or available_services
    if service_id not in available_services and available_services:
        service_id = available_services[0]

    trips_f = trips[trips["service_id"] == service_id].copy()
    if trips_f.empty:
        trips_f = trips.copy()

    if "direction_id" not in trips_f.columns:
        trips_f["direction_id"] = 0
    trips_f["direction_id"] = (
        pd.to_numeric(trips_f["direction_id"], errors="coerce").fillna(0).astype(int)
    )

    first_stop = (
        stop_times.sort_values(["trip_id", "stop_sequence"])
        .groupby("trip_id", as_index=False)
        .first()[["trip_id", "departure_time", "arrival_time"]]
    )
    first_stop["dep_str"] = first_stop["departure_time"].fillna(first_stop["arrival_time"])
    first_stop["dep_min"] = first_stop["dep_str"].apply(_gtfs_time_to_minutes)
    first_stop = first_stop.dropna(subset=["dep_min"])

    merged = trips_f[["trip_id", "route_id", "direction_id"]].merge(
        first_stop[["trip_id", "dep_min"]], on="trip_id", how="inner"
    )
    merged["hour"] = (merged["dep_min"] // 60).astype(int) % 24
    merged["route_id"] = merged["route_id"].astype(str)

    rows = []
    for (route, direction, hour), g in merged.groupby(["route_id", "direction_id", "hour"]):
        times = np.sort(g["dep_min"].to_numpy() % (24 * 60))
        n = int(len(times))
        if n <= 0:
            continue
        avg_hw = 60.0 / n
        if n >= 2:
            deltas = np.diff(times)
            deltas = deltas[deltas > 0]
            min_hw = float(np.min(deltas)) if deltas.size else avg_hw
        else:
            min_hw = avg_hw
        rows.append({
            "route_id": route,
            "direction_id": int(direction),
            "hour": int(hour),
            "trip_count": n,
            "avg_headway_min": float(avg_hw),
            "min_headway_min": float(min_hw),
        })

    hw = pd.DataFrame(rows).sort_values(["route_id", "direction_id", "hour"]).reset_index(drop=True)
    hw.attrs["service_id"] = service_id
    return hw


factors = load_factors_hourly(FACTORS_HOURLY_FILE)
print("Factors: hourly", factors.shape, "| cols:", len(factors.columns))

headway = build_headway_from_gtfs(SCHEDULE_DIR, service_id=HEADWAY_SERVICE)
headway["route_id"] = headway["route_id"].astype(str)
headway.to_csv(OUT_DIR / "headway_by_route_hour_derived.csv", index=False)

print(
    f"Headway dựng từ GTFS (service={headway.attrs.get('service_id', HEADWAY_SERVICE)}):",
    headway.shape,
)
print("Hours in headway:", sorted(headway["hour"].unique()))
print("Routes in headway:", headway["route_id"].nunique())
factors.head(3)


In [ ]:
ridership_route["date"] = pd.to_datetime(ridership_route["date"]).dt.normalize()
ridership_route["hour"] = ridership_route["hour"].astype(int)


ridership_all = ridership_route.merge(factors, on=FACTORS_JOIN_KEYS, how="left")
if "day_of_week" in ridership_all.columns:
    ridership_all["day_of_week"] = ridership_all["day_of_week"].fillna(
        ridership_all["date"].dt.dayofweek
    )
else:
    ridership_all["day_of_week"] = ridership_all["date"].dt.dayofweek
if "is_weekend" in ridership_all.columns:
    ridership_all["is_weekend"] = ridership_all["is_weekend"].fillna(
        (ridership_all["date"].dt.dayofweek >= 5).astype(int)
    )
else:
    ridership_all["is_weekend"] = (ridership_all["date"].dt.dayofweek >= 5).astype(int)
miss = ridership_all["temperature_c"].isna().mean() if "temperature_c" in ridership_all.columns else 1.0
print(f"Merge hourly on {FACTORS_JOIN_KEYS}: missing weather rows = {miss:.2%}")

num_cols = [
    "temperature_c", "apparent_temperature_c",
    "precipitation_mm", "rain_mm", "snowfall_cm",
    "windspeed_kmh", "windgusts_kmh",
]
flag_cols = [
    "is_us_holiday", "is_rain", "is_snow", "is_severe_wind",
    "is_peak_morning", "is_peak_evening", "is_overnight",
    "is_major_event_window",
]

for c in num_cols:
    if c not in ridership_all.columns:
        ridership_all[c] = 0.0
    ridership_all[c] = ridership_all[c].fillna(ridership_all[c].median())
for c in flag_cols:
    if c not in ridership_all.columns:
        ridership_all[c] = 0
    ridership_all[c] = ridership_all[c].fillna(0).astype(int)

print("Merged ridership shape:", ridership_all.shape)
print("Hours:", sorted(ridership_all["hour"].unique()))
ridership_all.head()


## 3. EDA & Chọn scope tuyến

Mục tiêu: mô tả demand theo giờ/tuyến, chọn tập tuyến tối ưu (`OPT_ROUTE_MODE`, mặc định `headway`).

- **Full run (GTFS giờ thực tế)**: 29 tuyến `OPT_ROUTES` → **1.182 slot** sau lọc giờ phục vụ theo `(route, direction)` (lý thuyết 29×2×24 = 1.392).
- **NN dataset**: 399.840 bản ghi, **25 tuyến** có đủ ridership sau lọc giờ; date range **2023-12 → 2025-11** (731 ngày).
- **Headway fallback** (run hiện tại): **0/1.182** slot — baseline GTFS exact cho mọi slot trong scope.
- EDA heatmap/top tuyến: demand cao nhất **1, 6, 7, R, F**; peak 7–9h và 17–19h.


In [ ]:
HEADWAY_ROUTES = sorted(headway["route_id"].astype(str).unique())
route_row_counts = ridership_all.groupby("route_id").size()
RIDERSHIP_ROUTES = route_row_counts[route_row_counts >= MIN_ROUTE_DEMAND_ROWS].index.astype(str).tolist()

if OPT_ROUTE_MODE == "headway":
    OPT_ROUTES = HEADWAY_ROUTES
elif OPT_ROUTE_MODE == "intersection":
    OPT_ROUTES = sorted(set(HEADWAY_ROUTES) & set(RIDERSHIP_ROUTES))
else:
    OPT_ROUTES = RIDERSHIP_ROUTES

route_totals = (
    ridership_all.groupby("route_id")["demand"].mean().sort_values(ascending=False)
)
print("Top 12 tuyến theo demand TB/giờ:")
print(route_totals.head(12).round(0))
print()
print(f"OPT_ROUTES ({len(OPT_ROUTES)} tuyến, mode={OPT_ROUTE_MODE}):", OPT_ROUTES[:8], "...")

route_hours = {r: list(OPT_HOURS_FALLBACK) for r in OPT_ROUTES}
route_dir_hours: dict[tuple[str, int], list[int]] = {}
if USE_GTFS_OPERATING_HOURS:
    route_dir_hours = srp.derive_operating_hours_by_route_direction(
        headway, OPT_ROUTES, directions=DIRECTIONS, min_trips=MIN_GTFS_TRIPS_PER_HOUR
    )
    route_hours = srp.derive_operating_hours_by_route(
        headway, OPT_ROUTES, directions=DIRECTIONS, min_trips=MIN_GTFS_TRIPS_PER_HOUR
    )
    empty = [k for k, hs in route_dir_hours.items() if not hs]
    if empty:
        raise ValueError(f"Không có giờ GTFS cho: {empty[:8]}")
    OPT_HOURS = sorted({h for hs in route_dir_hours.values() for h in hs})
    print(f"GTFS operating hours (route×dir): {len(route_dir_hours)} keys, union {len(OPT_HOURS)} hours")
    lens = [len(hs) for hs in route_dir_hours.values()]
    print(f"  hours per route×dir: min={min(lens)}, max={max(lens)}")

OPT_SCOPE = [
    (r, d, h) for r in OPT_ROUTES for d in DIRECTIONS
    for h in (route_dir_hours.get((r, d)) or route_hours.get(r, OPT_HOURS_FALLBACK))
]
print(f"OPT_SCOPE slots (pre-filter ridership): {len(OPT_SCOPE)}")

df = ridership_all[ridership_all["route_id"].isin(OPT_ROUTES)].copy()
n_all = len(df)
if USE_GTFS_OPERATING_HOURS:
    df = df[df.apply(lambda row: row["hour"] in route_hours.get(row["route_id"], []), axis=1)].copy()
print()
print(f"Dataset NN: {len(df):,} bản ghi ({n_all:,} trước lọc giờ), "
      f"{df['date'].nunique()} ngày, routes={df['route_id'].nunique()}")
print(f"Demand range: {df['demand'].min():.0f} – {df['demand'].max():.0f}")


## 3b. Transfer network & demand spillover

**Bước 1**: `build_transfer_matrix` — cosine-like weight từ `routes_by_station_complex`.
**Bước 2**: `adjust_demand_with_spillover` áp dụng trước optimize (không đổi solver).
**Bước 3**: Validate MAE hold-out + heatmap + tuyến benefit nhất (§4a).

In [ ]:
# BƯỚC 1 — Transfer adjacency (route × route)
transfer_matrix = srp.build_transfer_matrix(
    routes_complex,
    OPT_ROUTES,
    route_aliases=ROUTE_ALIASES,
    min_weight=TRANSFER_MIN_WEIGHT,
)
_off_diag = ((transfer_matrix > 0).values) & (~np.eye(len(transfer_matrix), dtype=bool))
_n_off = int(_off_diag.sum())
print(f"Transfer matrix: {transfer_matrix.shape[0]} routes | "
      f"nonzero pairs={int((transfer_matrix > 0).sum().sum())} | "
      f"off-diag connections={_n_off}")
print("Top connectivity (sum weight):")
print(transfer_matrix.sum(axis=1).sort_values(ascending=False).head(8).round(2).to_string())
transfer_matrix.to_csv(OUT_DIR / "transfer_matrix.csv")
print("Đã lưu:", OUT_DIR / "transfer_matrix.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

heat = df.groupby(["route_id", "hour"])["demand"].mean().unstack()
_routes_in_df = set(heat.index.astype(str))
top_plot = [r for r in route_totals.index.astype(str) if r in _routes_in_df][:12]
heat = heat.reindex(top_plot).dropna(how="all")
sns.heatmap(heat, ax=axes[0], cmap="YlOrRd", annot=False)
axes[0].set_title(f"Demand TB: {len(top_plot)} tuyến × giờ (trong scope)")

wk = df.pivot_table(index="hour", columns="is_weekend", values="demand", aggfunc="mean")
wk = wk.rename(columns={0: "Weekday", 1: "Weekend"})
wk.plot(kind="bar", ax=axes[1], color=["#3498DB", "#E74C3C"])
axes[1].set_title("Demand TB theo giờ: Weekday vs Weekend")
axes[1].set_xlabel("Hour"); axes[1].legend()

rain_col = "is_rain" if "is_rain" in df.columns else "is_rainy_day"
rain = df.pivot_table(index="hour", columns=rain_col, values="demand", aggfunc="mean")
rain = rain.rename(columns={0: "Clear", 1: "Rainy"})
rain.plot(kind="bar", ax=axes[2], color=["#27AE60", "#2C3E50"])
axes[2].set_title("Demand TB: Clear vs Rainy (theo giờ)")
axes[2].set_xlabel("Hour"); axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "fig_eda.png", dpi=120, bbox_inches="tight")
plt.show()


**Quan sát EDA** (full run hiện tại):

- Demand biến thiên mạnh theo giờ; peak 7–9h và 17–19h nổi bật trên tuyến trục (1, 6, 7).
- **Weekday vs weekend**: cuối tuần lệch khung giờ đi làm so ngày thường.
- **Clear vs rainy** (proxy `is_rain`): kịch bản `rainy_day` dùng để kiểm thử NN + optimizer; tỷ lệ rainy/weekday_peak TB ≈ theo output cell kịch bản.
- `OPT_SCOPE` pre-filter: **1.182 slots**; sau merge ridership còn **432.615** (route,date,hour) records, coverage ≥ 0.7.


In [ ]:
df = df.sort_values(["route_id", "date", "hour"]).reset_index(drop=True)

if "month" not in df.columns:
    df["month"] = df["date"].dt.month

df = srp.add_cyclical_time_features(df)
df = srp.add_weather_interaction_features(df)

route_to_idx = {r: i for i, r in enumerate(sorted(OPT_ROUTES))}
df["route_idx"] = df["route_id"].map(route_to_idx).astype(int)

df = srp.add_lag_features(df, use_lags=USE_LAG_FEATURES, lag_cols=LAG_FEATURE_COLS)

unique_dates = sorted(df["date"].unique())
n_dates = len(unique_dates)

train_dates, val_dates, test_dates, holdout_meta = srp.build_seasonal_holdout_splits(
    unique_dates,
    test_season=HOLDOUT_TEST_SEASON,
    test_season_year=HOLDOUT_TEST_YEAR,
    val_season=HOLDOUT_VAL_SEASON,
    val_season_year=HOLDOUT_VAL_YEAR,
    auto_adjust=False,
    return_meta=True,
)
HOLDOUT_TEST_SEASON = holdout_meta["test_season"]
HOLDOUT_TEST_YEAR = holdout_meta["test_season_year"]
HOLDOUT_VAL_SEASON = holdout_meta["val_season"]
HOLDOUT_VAL_YEAR = holdout_meta["val_season_year"]

train_df = df[df["date"].isin(train_dates)].copy()
val_df   = df[df["date"].isin(val_dates)].copy()
test_df  = df[df["date"].isin(test_dates)].copy()

baseline_lookup = (
    train_df.groupby(["route_id", "hour", "is_weekend"])["demand"]
    .median()
    .rename("baseline_demand")
)
fallback = train_df.groupby(["route_id", "hour"])["demand"].median().rename("fallback_demand")


def attach_baseline(d: pd.DataFrame) -> pd.DataFrame:
    d = d.merge(baseline_lookup, on=["route_id", "hour", "is_weekend"], how="left")
    d = d.merge(fallback, on=["route_id", "hour"], how="left")
    d["baseline_demand"] = d["baseline_demand"].fillna(d["fallback_demand"])
    return d.drop(columns=["fallback_demand"])


train_df = attach_baseline(train_df)
val_df   = attach_baseline(val_df)
test_df  = attach_baseline(test_df)

train_df = srp.fill_lag_from_train(train_df, train_df, lag_cols=LAG_FEATURE_COLS)
val_df   = srp.fill_lag_from_train(val_df, train_df, lag_cols=LAG_FEATURE_COLS)
test_df  = srp.fill_lag_from_train(test_df, train_df, lag_cols=LAG_FEATURE_COLS)

for part in (train_df, val_df, test_df):
    part["log_baseline"] = np.log1p(part["baseline_demand"])

NUM_FEATURES = srp.build_num_feature_list(
    HOURLY_FACTOR_COLS,
    use_lags=USE_LAG_FEATURES,
    lag_cols=LAG_FEATURE_COLS,
    extra_interactions=True,
)
NUM_FEATURES = [c for c in NUM_FEATURES if c in train_df.columns]
print(f"NUM_FEATURES ({len(NUM_FEATURES)}):", NUM_FEATURES)

lag_feature_defaults = {}
if USE_LAG_FEATURES:
    lag_med = train_df.groupby(["route_id", "hour"])[LAG_FEATURE_COLS].median()
    for (rid, hour), row in lag_med.iterrows():
        lag_feature_defaults[f"{rid}|{hour}"] = {c: float(row[c]) for c in LAG_FEATURE_COLS}

print(f"Hold-out theo mùa: test={HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR}, "
      f"val={HOLDOUT_VAL_SEASON}_{HOLDOUT_VAL_YEAR}")
print(f"{n_dates} ngày → train {len(train_dates)} / val {len(val_dates)} / test {len(test_dates)}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

y_train = np.log1p(train_df["demand"].values) - train_df["log_baseline"].values
y_val   = np.log1p(val_df["demand"].values) - val_df["log_baseline"].values
y_test  = np.log1p(test_df["demand"].values) - test_df["log_baseline"].values

scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_df[NUM_FEATURES])
X_val_num   = scaler.transform(val_df[NUM_FEATURES])
X_test_num  = scaler.transform(test_df[NUM_FEATURES])
X_train_route = train_df["route_idx"].values.reshape(-1, 1)
X_val_route   = val_df["route_idx"].values.reshape(-1, 1)
X_test_route  = test_df["route_idx"].values.reshape(-1, 1)

print("Target (residual log demand) — train mean: {:.3f}, std: {:.3f}".format(
    y_train.mean(), y_train.std()))


In [ ]:
lstm_model = None
histgbm_model = None
blend_weight_mlp = 0.5
blend_weight_tuned = False


def make_demand_model(n_routes: int, n_num_features: int) -> Model:
    return srp.build_demand_model(
        n_routes,
        n_num_features,
        use_route_embedding=USE_ROUTE_EMBEDDING,
        hidden=NN_HIDDEN,
        dropout=NN_DROPOUT,
        use_batch_norm=USE_BATCH_NORM,
    )


def _fit_inputs(route_idx, num_feat):
    return srp.format_keras_inputs(
        route_idx, num_feat, use_route_embedding=USE_ROUTE_EMBEDDING
    )


if DEMAND_MODEL_TYPE == "lstm":
    train_seq = pd.concat([train_df, val_df], ignore_index=True)
    X_seq, y_seq, _ = srp.prepare_lstm_sequences(
        train_seq, NUM_FEATURES, seq_len=LSTM_SEQ_LEN, target_residual=True
    )
    scaler_lstm = StandardScaler()
    n_s, n_t, n_f = X_seq.shape
    X_seq_s = scaler_lstm.fit_transform(X_seq.reshape(-1, n_f)).reshape(n_s, n_t, n_f)
    lstm_model = srp.build_lstm_demand_model(LSTM_SEQ_LEN, n_f)
    lstm_model.summary()
    model = lstm_model
else:
    model = make_demand_model(len(route_to_idx), len(NUM_FEATURES))
    model.summary()


In [ ]:
train_weights = np.ones(len(train_df), dtype=float)
if PEAK_SAMPLE_WEIGHT > 1.0:
    peak_mask = (train_df["is_peak_morning"] == 1) | (train_df["is_peak_evening"] == 1)
    train_weights[peak_mask.to_numpy()] = PEAK_SAMPLE_WEIGHT
    print(f"Peak sample weight ×{PEAK_SAMPLE_WEIGHT}: {peak_mask.sum():,}/{len(train_df):,} rows")

early = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)
lr_reduce = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5)

if DEMAND_MODEL_TYPE == "lstm":
    history = model.fit(
        X_seq_s, y_seq,
        epochs=NN_EPOCHS_MAIN,
        batch_size=32,
        verbose=0,
        validation_split=0.15,
        callbacks=[early, lr_reduce],
    )
else:
    history = model.fit(
        _fit_inputs(X_train_route, X_train_num),
        y_train,
        validation_data=(_fit_inputs(X_val_route, X_val_num), y_val),
        sample_weight=train_weights,
        epochs=NN_EPOCHS_MAIN,
        batch_size=32,
        verbose=0,
        callbacks=[early, lr_reduce],
    )
print(f"Stop @ epoch {len(history.history['loss'])} | best val_mae={min(history.history.get('val_mae', history.history['loss'])):.4f}")

if USE_HISTGBM_BLEND and DEMAND_MODEL_TYPE in ("blend", "mlp"):
    histgbm_model = srp.fit_histgbm_demand(X_train_num, y_train, sample_weight=train_weights)
    print("HistGBM (literature tree baseline) fitted.")


In [ ]:
if TUNE_RESID_CLIP_ON_VAL and DEMAND_MODEL_TYPE != "lstm":
    best_clip, best_mae = RESID_CLIP[1], float("inf")
    for clip_hi in RESID_CLIP_CANDIDATES:
        tmp_clip = (-clip_hi, clip_hi)
        pred_resid = model.predict(_fit_inputs(X_val_route, X_val_num), verbose=0).ravel()
        pred_resid = np.clip(pred_resid, *tmp_clip)
        pred_val = np.expm1(val_df["log_baseline"].values + pred_resid)
        mae_val = mean_absolute_error(val_df["demand"].values, pred_val)
        print(f"  val MAE @ clip ±{clip_hi:.2f}: {mae_val:,.1f}")
        if mae_val < best_mae:
            best_mae, best_clip = mae_val, clip_hi
    RESID_CLIP = (-best_clip, best_clip)
    print(f"→ Chọn RESID_CLIP = ±{best_clip:.2f} (val MAE={best_mae:,.1f})")


def _predict_resid_mlp(route_idx_arr, num_feat_arr):
    return model.predict(_fit_inputs(route_idx_arr, num_feat_arr), verbose=0).ravel()


blend_weight_tuned = False


def tune_blend_weight_mlp() -> None:
    """Chọn trọng số MLP/HistGBM trên val — chỉ chạy một lần."""
    global blend_weight_mlp, blend_weight_tuned
    if DEMAND_MODEL_TYPE != "blend" or histgbm_model is None:
        return
    if blend_weight_tuned:
        return
    if HISTGBM_BLEND_WEIGHT is not None:
        blend_weight_mlp = float(HISTGBM_BLEND_WEIGHT)
        blend_weight_tuned = True
        return
    pred_mlp = np.clip(_predict_resid_mlp(X_val_route, X_val_num), *RESID_CLIP)
    pred_gbm = histgbm_model.predict(X_val_num)
    best_w, best_mae = 0.5, float("inf")
    for w in np.linspace(0.2, 0.8, 7):
        pr = srp.blend_residual_predictions(pred_mlp, pred_gbm, w)
        pv = srp.residuals_to_demand(val_df["log_baseline"].values, pr, RESID_CLIP)
        m = mean_absolute_error(val_df["demand"].values, pv)
        if m < best_mae:
            best_mae, best_w = m, w
    blend_weight_mlp = best_w
    blend_weight_tuned = True
    print(f"Blend val: MLP weight={blend_weight_mlp:.2f} (MAE={best_mae:,.1f})")


def predict_demand(route_idx_arr, num_feat_arr, log_baseline, frame: pd.DataFrame | None = None):
    if DEMAND_MODEL_TYPE == "lstm" and frame is not None:
        X_te, _, meta = srp.prepare_lstm_sequences(
            frame, NUM_FEATURES, seq_len=LSTM_SEQ_LEN, target_residual=True
        )
        n_s, n_t, n_f = X_te.shape
        X_te_s = scaler_lstm.transform(X_te.reshape(-1, n_f)).reshape(n_s, n_t, n_f)
        pred_resid = model.predict(X_te_s, verbose=0).ravel()
        pred_log = meta["log_baseline"].values + np.clip(pred_resid, *RESID_CLIP)
        out = pd.Series(np.expm1(pred_log), index=meta.index)
        return out.reindex(frame.index, method=None).fillna(np.expm1(log_baseline)).to_numpy()

    pred_mlp = np.clip(_predict_resid_mlp(route_idx_arr, num_feat_arr), *RESID_CLIP)
    if histgbm_model is not None and DEMAND_MODEL_TYPE in ("blend", "mlp"):
        pred_gbm = histgbm_model.predict(num_feat_arr)
        if DEMAND_MODEL_TYPE == "blend":
            tune_blend_weight_mlp()
            pred_resid = srp.blend_residual_predictions(pred_mlp, pred_gbm, blend_weight_mlp)
        else:
            pred_resid = pred_gbm
    else:
        pred_resid = pred_mlp
    return srp.residuals_to_demand(log_baseline, pred_resid, RESID_CLIP)


if DEMAND_MODEL_TYPE == "lstm":
    y_pred = predict_demand(None, None, test_df["log_baseline"].values, frame=test_df)
else:
    y_pred = predict_demand(X_test_route, X_test_num, test_df["log_baseline"].values)

y_true = test_df["demand"].values
baseline_pred_test = test_df["baseline_demand"].values
m_base = srp.regression_metrics(y_true, baseline_pred_test)
m_nn = srp.regression_metrics(y_true, y_pred)
mae_impr = (m_base["mae"] - m_nn["mae"]) / m_base["mae"] * 100

nn_eval_metrics = {
    "holdout": {
        "test_block": f"{HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR}",
        "val_block": f"{HOLDOUT_VAL_SEASON}_{HOLDOUT_VAL_YEAR}",
        "n_test": int(len(test_df)),
        "model_type": DEMAND_MODEL_TYPE,
        "blend_mlp_weight": float(blend_weight_mlp) if histgbm_model is not None else None,
        "baseline": m_base,
        "model": m_nn,
        "mae_improvement_pct": float(mae_impr),
        "resid_clip": float(RESID_CLIP[1]),
    },
}

def _fmt(m):
    return f"MAE={m['mae']:,.0f} RMSE={m['rmse']:,.0f} R²={m['r2']:+.3f} MAPE={m['mape_pct']:.1f}% SMAPE={m['smape_pct']:.1f}%"

print("=== Hold-out test ===")
print(f"  {DEMAND_MODEL_TYPE} | test={HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR}")
print(f"  Baseline: {_fmt(m_base)}")
print(f"  Model:    {_fmt(m_nn)}  (ΔMAE {mae_impr:+.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
if "val_loss" in history.history:
    axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title(f"Loss ({DEMAND_MODEL_TYPE})")
axes[0].legend()
axes[1].scatter(y_true, y_pred, alpha=0.7, s=22, label=f"Model R²={m_nn['r2']:.2f}")
axes[1].scatter(y_true, baseline_pred_test, alpha=0.4, s=18, marker="x", label=f"Baseline R²={m_base['r2']:.2f}")
lim = max(y_true.max(), y_pred.max())
axes[1].plot([0, lim], [0, lim], "r--", lw=1)
axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_nn_eval.png", dpi=120, bbox_inches="tight")
plt.show()



## 4a. Validate demand spillover (hold-out autumn_2025)

So sánh MAE forecast **trước / sau** `adjust_demand_with_spillover`; heatmap transfer matrix; tuyến benefit nhất.

In [ ]:
# BƯỚC 3 — Validate spillover trên hold-out test
if DEMAND_MODEL_TYPE == "lstm":
    print("Spillover validation: bỏ qua cho LSTM.")
else:
    _hours_sorted = sorted(test_df["hour"].unique())
    _test_agg = (
        test_df.assign(y_pred=y_pred)
        .groupby(["route_id", "hour"], as_index=False)
        .agg(demand_actual=("demand", "mean"), demand_pred=("y_pred", "mean"))
    )
    _routes_val = [r for r in OPT_ROUTES if r in _test_agg["route_id"].values]
    _actual_dict = srp.route_hour_df_to_demand_dict(
        _test_agg.rename(columns={"demand_actual": "demand"}),
        _routes_val, _hours_sorted, demand_col="demand",
    )
    _pred_dict = srp.route_hour_df_to_demand_dict(
        _test_agg.rename(columns={"demand_pred": "demand"}),
        _routes_val, _hours_sorted, demand_col="demand",
    )

    _alpha_rows = []
    for a in SPILLOVER_ALPHA_GRID:
        m = srp.evaluate_spillover_forecast_mae(
            _actual_dict, _pred_dict, transfer_matrix, alpha=a
        )
        _alpha_rows.append(m)
    spillover_alpha_scan = pd.DataFrame(_alpha_rows)
    _best = spillover_alpha_scan.loc[spillover_alpha_scan["mae_after"].idxmin()]
    print("=== Spillover MAE (hold-out route×hour mean) ===")
    print(spillover_alpha_scan.round(2).to_string(index=False))
    print(f"Best alpha={_best['alpha']:.2f} | MAE {_best['mae_before']:.1f} → {_best['mae_after']:.1f} "
          f"(Δ={_best['mae_delta']:+.1f})")

    spillover_benefit = srp.spillover_benefit_by_route(
        _actual_dict, _pred_dict, transfer_matrix, alpha=float(SPILLOVER_ALPHA)
    )
    print(f"\nTop tuyến benefit nhất @ alpha={SPILLOVER_ALPHA}:")
    print(spillover_benefit.head(10).round(2).to_string(index=False))
    print("Bottom (spillover làm tệ hơn):")
    print(spillover_benefit.tail(5).round(2).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    srp.plot_transfer_heatmap(
        transfer_matrix, top_n=10, ax=axes[0],
        title="Transfer matrix (top 10 connectivity)",
    )
    axes[1].barh(
        spillover_benefit.head(12)["route_id"],
        spillover_benefit.head(12)["benefit"],
        color=np.where(spillover_benefit.head(12)["benefit"] >= 0, "#2ca25f", "#de2d26"),
    )
    axes[1].axvline(0, color="k", lw=0.8)
    axes[1].set_xlabel("MAE improvement (before − after)")
    axes[1].set_title(f"Top routes benefit @ α={SPILLOVER_ALPHA}")
    axes[1].invert_yaxis()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_transfer_spillover.png", dpi=120, bbox_inches="tight")
    plt.show()

    spillover_alpha_scan.to_csv(OUT_DIR / "spillover_alpha_scan.csv", index=False)
    spillover_benefit.to_csv(OUT_DIR / "spillover_benefit_by_route.csv", index=False)
    print("Đã lưu:", OUT_DIR / "spillover_alpha_scan.csv", "|", OUT_DIR / "fig_transfer_spillover.png")

In [ ]:
if DEMAND_MODEL_TYPE == "lstm":
    print("CV mùa×năm: bỏ qua cho LSTM — chỉ dùng hold-out.")
    nn_cv_summary = {"cv_mode": "skipped_lstm", "n_folds": 0}
    nn_eval_metrics["cv"] = nn_cv_summary
else:
    all_df = df.copy()
    if "month" not in all_df.columns:
        all_df["month"] = all_df["date"].dt.month
    all_df = srp.add_cyclical_time_features(all_df)
    all_df = srp.add_weather_interaction_features(all_df)
    all_df = srp.add_lag_features(all_df, use_lags=USE_LAG_FEATURES, lag_cols=LAG_FEATURE_COLS)
    all_df = attach_baseline(all_df)
    all_df = srp.fill_lag_from_train(all_df, train_df, lag_cols=LAG_FEATURE_COLS)
    all_df["log_baseline"] = np.log1p(all_df["baseline_demand"])

    dates_arr = np.array(sorted(all_df["date"].unique()))
    season_cv_folds = srp.build_season_year_cv_folds(dates_arr, n_years=CV_SEASON_YEARS)
    if not season_cv_folds:
        raise ValueError("Không đủ dữ liệu cho CV mùa×năm — cần ít nhất một ngày mỗi (mùa, năm)")
    print(f"CV theo mùa×năm: {len(season_cv_folds)} fold (kỳ vọng tối đa {CV_SPLITS})")

    cv_mae_nn, cv_mae_base, cv_rmse_nn, cv_rmse_base, cv_r2_nn, cv_r2_base = [], [], [], [], [], []
    for fold, (tr_dates, te_dates, fold_label) in enumerate(season_cv_folds):
        tr_d = all_df[all_df["date"].isin(tr_dates)].copy()
        te_d = all_df[all_df["date"].isin(te_dates)].copy()
        if len(tr_d) == 0 or len(te_d) == 0:
            continue

        bl_lookup = (
            tr_d.groupby(["route_id", "hour", "is_weekend"])["demand"].median()
            .rename("baseline_demand_cv")
        )
        fb = tr_d.groupby(["route_id", "hour"])["demand"].median().rename("fb_demand")
        te_d = te_d.drop(columns=["baseline_demand", "log_baseline"], errors="ignore") \
                    .merge(bl_lookup, on=["route_id", "hour", "is_weekend"], how="left") \
                    .merge(fb, on=["route_id", "hour"], how="left")
        te_d["baseline_demand_cv"] = te_d["baseline_demand_cv"].fillna(te_d["fb_demand"])
        te_d = te_d.dropna(subset=["baseline_demand_cv"])
        te_d["log_baseline"] = np.log1p(te_d["baseline_demand_cv"])

        tr_d2 = tr_d.drop(columns=["baseline_demand", "log_baseline"], errors="ignore") \
                    .merge(bl_lookup, on=["route_id", "hour", "is_weekend"], how="left") \
                    .merge(fb, on=["route_id", "hour"], how="left")
        tr_d2["baseline_demand_cv"] = tr_d2["baseline_demand_cv"].fillna(tr_d2["fb_demand"])
        tr_d2 = tr_d2.dropna(subset=["baseline_demand_cv"])
        tr_d2["log_baseline"] = np.log1p(tr_d2["baseline_demand_cv"])

        sc = StandardScaler()
        Xt = sc.fit_transform(tr_d2[NUM_FEATURES])
        Xe = sc.transform(te_d[NUM_FEATURES])
        yt = np.log1p(tr_d2["demand"].values) - tr_d2["log_baseline"].values
        ye = te_d["demand"].values

        w_cv = np.ones(len(tr_d2), dtype=float)
        if PEAK_SAMPLE_WEIGHT > 1.0:
            pk = (tr_d2["is_peak_morning"] == 1) | (tr_d2["is_peak_evening"] == 1)
            w_cv[pk.to_numpy()] = PEAK_SAMPLE_WEIGHT

        sorted_tr = sorted(tr_dates)
        n_tune = max(1, int(len(sorted_tr) * CV_BLEND_TUNE_FRAC))
        tune_dates = set(sorted_tr[-n_tune:])
        fit_mask = ~tr_d2["date"].isin(tune_dates)
        tune_mask = tr_d2["date"].isin(tune_dates)

        m = make_demand_model(len(route_to_idx), len(NUM_FEATURES))
        m.fit(
            _fit_inputs(tr_d2.loc[fit_mask, "route_idx"].values.reshape(-1, 1), Xt[fit_mask]),
            yt[fit_mask],
            sample_weight=w_cv[fit_mask.to_numpy()],
            epochs=NN_EPOCHS_CV,
            batch_size=32,
            verbose=0,
            callbacks=[EarlyStopping(monitor="loss", patience=15, restore_best_weights=True)],
        )

        blend_w = 0.5
        hist_cv = None
        use_blend_cv = CV_USE_BLEND and USE_HISTGBM_BLEND and DEMAND_MODEL_TYPE == "blend"
        if use_blend_cv:
            hist_cv = srp.fit_histgbm_demand(Xt[fit_mask], yt[fit_mask], sample_weight=w_cv[fit_mask.to_numpy()])
            pred_mlp_tune = m.predict(
                _fit_inputs(tr_d2.loc[tune_mask, "route_idx"].values.reshape(-1, 1), Xt[tune_mask]),
                verbose=0,
            ).ravel()
            pred_gbm_tune = hist_cv.predict(Xt[tune_mask])
            blend_w, _ = srp.tune_blend_weight_mae(
                pred_mlp_tune,
                pred_gbm_tune,
                tr_d2.loc[tune_mask, "log_baseline"].values,
                tr_d2.loc[tune_mask, "demand"].values,
                RESID_CLIP,
            )

        pred_mlp = m.predict(_fit_inputs(te_d["route_idx"].values.reshape(-1, 1), Xe), verbose=0).ravel()
        if use_blend_cv and hist_cv is not None:
            pred_resid = srp.blend_residual_predictions(pred_mlp, hist_cv.predict(Xe), blend_w)
        else:
            pred_resid = pred_mlp
        pred_resid = np.clip(pred_resid, *RESID_CLIP)
        pred_nn = np.expm1(te_d["log_baseline"].values + pred_resid)
        pred_base = te_d["baseline_demand_cv"].values

        mb = srp.regression_metrics(ye, pred_base)
        mn = srp.regression_metrics(ye, pred_nn)
        cv_mae_base.append(mb["mae"])
        cv_rmse_base.append(mb["rmse"])
        cv_r2_base.append(mb["r2"])
        cv_mae_nn.append(mn["mae"])
        cv_rmse_nn.append(mn["rmse"])
        cv_r2_nn.append(mn["r2"])
        tag = f"blend w={blend_w:.2f}" if use_blend_cv else DEMAND_MODEL_TYPE
        print(
            f"Fold {fold+1} ({fold_label}): train {len(tr_dates)} / test {len(te_dates)} dates  "
            f"| {tag} MAE={cv_mae_nn[-1]:,.0f} RMSE={cv_rmse_nn[-1]:,.0f} R²={cv_r2_nn[-1]:+.2f}  "
            f"| Baseline MAE={cv_mae_base[-1]:,.0f} RMSE={cv_rmse_base[-1]:,.0f}"
        )

    cv_impr = (np.mean(cv_mae_base) - np.mean(cv_mae_nn)) / np.mean(cv_mae_base) * 100
    print("\n=== 8-fold CV mùa×năm ===")
    print(
        f"NN       MAE={np.mean(cv_mae_nn):,.0f}±{np.std(cv_mae_nn):,.0f}  "
        f"RMSE={np.mean(cv_rmse_nn):,.0f}±{np.std(cv_rmse_nn):,.0f}  R²={np.mean(cv_r2_nn):+.3f}"
    )
    print(
        f"Baseline MAE={np.mean(cv_mae_base):,.0f}±{np.std(cv_mae_base):,.0f}  "
        f"RMSE={np.mean(cv_rmse_base):,.0f}±{np.std(cv_rmse_base):,.0f}"
    )
    print(f"NN cải thiện trung bình so baseline: {cv_impr:+.1f}% MAE")

    nn_cv_summary = {
        "cv_mode": "season_year",
        "cv_use_blend": bool(CV_USE_BLEND and USE_HISTGBM_BLEND and DEMAND_MODEL_TYPE == "blend"),
        "n_folds": len(cv_mae_nn),
        "mae_nn_mean": float(np.mean(cv_mae_nn)),
        "rmse_nn_mean": float(np.mean(cv_rmse_nn)),
        "mae_baseline_mean": float(np.mean(cv_mae_base)),
        "rmse_baseline_mean": float(np.mean(cv_rmse_base)),
        "r2_nn_mean": float(np.mean(cv_r2_nn)),
        "r2_baseline_mean": float(np.mean(cv_r2_base)),
        "mae_improvement_pct": float(cv_impr),
    }
    nn_eval_metrics["cv"] = nn_cv_summary

_cv_key = {
    "mae": ("mae_baseline_mean", "mae_nn_mean"),
    "rmse": ("rmse_baseline_mean", "rmse_nn_mean"),
    "r2": ("r2_baseline_mean", "r2_nn_mean"),
}
nn_report = pd.DataFrame(
    [
        {
            "metric": k,
            "holdout_baseline": nn_eval_metrics["holdout"]["baseline"][k],
            "holdout_nn": nn_eval_metrics["holdout"]["model"][k],
            "cv_mean_baseline": nn_cv_summary.get(_cv_key[k][0]) if DEMAND_MODEL_TYPE != "lstm" else None,
            "cv_mean_nn": nn_cv_summary.get(_cv_key[k][1]) if DEMAND_MODEL_TYPE != "lstm" else None,
        }
        for k in ("mae", "rmse", "r2")
    ]
)
print("\nBảng tổng hợp NN (hold-out vs CV):")
print(nn_report.round(3).to_string(index=False))
nn_report.to_csv(OUT_DIR / "nn_eval_summary.csv", index=False)
print("Đã lưu:", OUT_DIR / "nn_eval_summary.csv")


## 4b. Prediction intervals & uncertainty-aware scheduling

So sánh **Phương án A (MC Dropout MLP)** vs **Phương án B (Quantile HistGBM)** trên hold-out test:
- Calibration plot (nominal vs empirical coverage)
- Scatter `demand_std` vs `|forecast_error|`
- Interval width theo giờ (peak thường rộng hơn)
- Demo `uncertainty_aware_optimize` (aggressive / moderate / conservative)

In [ ]:
if DEMAND_MODEL_TYPE == "lstm":
    print("Uncertainty intervals: bỏ qua cho LSTM (chỉ MLP/HistGBM blend).")
else:
    tune_blend_weight_mlp()
    log_base_test = test_df["log_baseline"].values
    y_true_test = test_df["demand"].values
    hours_test = test_df["hour"].values

    # --- A: MC Dropout (MLP residual) ---
    mcd = srp.predict_with_uncertainty_mcdropout(
        model, _fit_inputs(X_test_route, X_test_num), n_samples=50, training=True
    )
    resid_mlp_mean = np.clip(mcd["mean_pred"], *RESID_CLIP)
    resid_mlp_std = mcd["std_pred"]
    if DEMAND_MODEL_TYPE == "blend" and histgbm_model is not None:
        resid_gbm = histgbm_model.predict(X_test_num)
        w = float(blend_weight_mlp)
        resid_a_mean = srp.blend_residual_predictions(resid_mlp_mean, resid_gbm, w)
        resid_a_std = w * resid_mlp_std
        resid_a_p05 = srp.blend_residual_predictions(
            np.clip(mcd["percentile_5"], *RESID_CLIP), resid_gbm, w
        )
        resid_a_p95 = srp.blend_residual_predictions(
            np.clip(mcd["percentile_95"], *RESID_CLIP), resid_gbm, w
        )
    else:
        resid_a_mean = resid_mlp_mean
        resid_a_std = resid_mlp_std
        resid_a_p05 = np.clip(mcd["percentile_5"], *RESID_CLIP)
        resid_a_p95 = np.clip(mcd["percentile_95"], *RESID_CLIP)

    unc_a = srp.residual_interval_to_demand(
        log_base_test, resid_a_mean,
        resid_low=resid_a_p05, resid_high=resid_a_p95, resid_std=resid_a_std, clip=RESID_CLIP,
    )
    pred_a = unc_a["demand_mean"]
    cal_a = srp.compute_interval_calibration(y_true_test, pred_a, unc_a["demand_std"])

    # --- B: Quantile HistGBM ---
    quantile_models = srp.fit_quantile_gbm(
        X_train_num, y_train,
        quantiles=[0.05, 0.5, 0.75, 0.95],
        sample_weight=train_weights,
    )
    qpred = srp.predict_quantile_gbm(quantile_models, X_test_num)
    resid_b_mean = np.clip(qpred["median"], *RESID_CLIP)
    resid_b_p05 = qpred["p05"]
    resid_b_p95 = qpred["p95"]
    resid_b_p75 = qpred.get("p75", qpred["median"])

    unc_b = srp.residual_interval_to_demand(
        log_base_test, resid_b_mean,
        resid_low=resid_b_p05, resid_high=resid_b_p95, clip=RESID_CLIP,
    )
    unc_b["demand_p75"] = np.expm1(log_base_test + np.asarray(resid_b_p75, dtype=float))
    unc_b["demand_std"] = (unc_b["demand_p95"] - unc_b["demand_p05"]) / (2 * 1.645)
    unc_b["interval_width"] = unc_b["demand_p95"] - unc_b["demand_p05"]
    pred_b = unc_b["demand_mean"]

    # Quantile calibration curve: các cặp (q_lo, q_hi) symmetric
    cal_b_rows = []
    for nom in np.linspace(0.5, 0.95, 10):
        q_lo = (1.0 - nom) / 2.0
        q_hi = 1.0 - q_lo
        if q_lo not in quantile_models:
            q_lo_m = min(quantile_models.keys(), key=lambda k: abs(k - q_lo))
            q_hi_m = min(quantile_models.keys(), key=lambda k: abs(k - q_hi))
        else:
            q_lo_m, q_hi_m = q_lo, q_hi
        lo = np.expm1(log_base_test + quantile_models[q_lo_m].predict(X_test_num))
        hi = np.expm1(log_base_test + quantile_models[q_hi_m].predict(X_test_num))
        cal_b_rows.append(dict(nominal=nom, empirical=float(np.mean((y_true_test >= lo) & (y_true_test <= hi)))))
    cal_b = pd.DataFrame(cal_b_rows)

    cov_a_90 = srp.compute_quantile_calibration(
        y_true_test, unc_a["demand_p05"], unc_a["demand_p95"], nominal=0.9
    )
    cov_b_90 = srp.compute_quantile_calibration(
        y_true_test, unc_b["demand_p05"], unc_b["demand_p95"], nominal=0.9
    )
    print("=== So sánh uncertainty (hold-out test) ===")
    print(f"  A MC Dropout  MAE={mean_absolute_error(y_true_test, pred_a):,.0f} | "
          f"90% coverage={cov_a_90['empirical']:.1%} (nominal 90%)")
    print(f"  B Quantile GBM MAE={mean_absolute_error(y_true_test, pred_b):,.0f} | "
          f"90% coverage={cov_b_90['empirical']:.1%} (nominal 90%)")
    print(f"  Mean interval width A={unc_a['interval_width'].mean():,.0f} | "
          f"B={unc_b['interval_width'].mean():,.0f}")

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    srp.plot_calibration_curve(
        cal_a, ax=axes[0, 0], label="A: MC Dropout",
        title="Calibration — MC Dropout (Gaussian)",
    )
    srp.plot_calibration_curve(
        cal_b, ax=axes[0, 1], label="B: Quantile GBM",
        title="Calibration — Quantile GBM",
    )
    srp.plot_std_vs_forecast_error(
        y_true_test, pred_a, unc_a["demand_std"], ax=axes[1, 0],
        title="A: std vs |error|",
    )
    srp.plot_std_vs_forecast_error(
        y_true_test, pred_b, unc_b["demand_std"], ax=axes[1, 1],
        title="B: std vs |error| (from q90−q10)",
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_uncertainty_calibration.png", dpi=120, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    srp.plot_interval_width_by_hour(
        hours_test, unc_a["interval_width"], ax=axes[0],
        title="A: interval width by hour",
    )
    srp.plot_interval_width_by_hour(
        hours_test, unc_b["interval_width"], ax=axes[1],
        title="B: interval width by hour",
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_interval_width_by_hour.png", dpi=120, bbox_inches="tight")
    plt.show()

    uncertainty_compare = pd.DataFrame([
        dict(method="mc_dropout", mae=mean_absolute_error(y_true_test, pred_a),
             coverage_90=cov_a_90["empirical"], mean_interval_width=unc_a["interval_width"].mean(),
             std_error_corr=float(np.corrcoef(unc_a["demand_std"], np.abs(y_true_test - pred_a))[0, 1])),
        dict(method="quantile_gbm", mae=mean_absolute_error(y_true_test, pred_b),
             coverage_90=cov_b_90["empirical"], mean_interval_width=unc_b["interval_width"].mean(),
             std_error_corr=float(np.corrcoef(unc_b["demand_std"], np.abs(y_true_test - pred_b))[0, 1])),
    ])
    uncertainty_compare.to_csv(OUT_DIR / "uncertainty_method_compare.csv", index=False)
    cal_a.to_csv(OUT_DIR / "calibration_mc_dropout.csv", index=False)
    cal_b.to_csv(OUT_DIR / "calibration_quantile_gbm.csv", index=False)
    print(uncertainty_compare.round(3).to_string(index=False))
    print("Đã lưu:", OUT_DIR / "fig_uncertainty_calibration.png")

    # --- Demo uncertainty-aware optimize (weekday_peak slots) ---
    # Map route×hour uncertainty từ test → slot (median test theo route×hour)
    _uh = pd.DataFrame({
        "route_id": test_df["route_id"].values,
        "hour": test_df["hour"].values,
        "demand_mean_a": pred_a,
        "demand_std_a": unc_a["demand_std"],
        "demand_p75_b": unc_b["demand_p75"],
        "demand_p05_b": unc_b["demand_p05"],
        "demand_p95_b": unc_b["demand_p95"],
    })
    _uh_med = _uh.groupby(["route_id", "hour"], as_index=False).median(numeric_only=True)
    _slot_u = pd.DataFrame({
        "route_id": slot_route, "hour": slot_hour,
        "demand_mean": scenario_demand["weekday_peak"],
    }).merge(_uh_med, on=["route_id", "hour"], how="left")
    for c in ["demand_mean_a", "demand_std_a", "demand_p75_b", "demand_p05_b", "demand_p95_b"]:
        _slot_u[c] = _slot_u[c].fillna(_slot_u["demand_mean"])

    d_mean = _slot_u["demand_mean"].to_numpy()
    d_std = _slot_u["demand_std_a"].fillna(0.1 * d_mean).to_numpy()
    d_p75 = _slot_u["demand_p75_b"].to_numpy()
    d_p05 = _slot_u["demand_p05_b"].to_numpy()
    d_p95 = _slot_u["demand_p95_b"].to_numpy()

    ua_rows = []
    for risk in ("aggressive", "moderate", "conservative"):
        trips_opt, (trips_lo, trips_hi) = srp.uncertainty_aware_optimize(
            d_mean, d_std, TRIPS_MIN, TRIPS_MAX, LAMBDA_COST,
            risk_level=risk,
            demand_p75=d_p75, demand_p05=d_p05, demand_p95=d_p95,
        )
        m = evaluate_schedule(trips_opt, d_mean, lambda_cost=LAMBDA_COST)
        ua_rows.append(dict(
            risk_level=risk, total_trips=m["total_trips"],
            total_wait=m["total_passenger_min_wait"],
            trips_band_width=float((trips_hi - trips_lo).mean()),
        ))
    ua_df = pd.DataFrame(ua_rows)
    print("\n=== uncertainty_aware_optimize (weekday_peak) ===")
    print(ua_df.round(1).to_string(index=False))
    ua_df.to_csv(OUT_DIR / "uncertainty_aware_schedule.csv", index=False)

**Tổng kết Neural Network** (full run `default`, blend MLP+HistGBM):

| | MAE | RMSE | R² | MAPE |
|--|-----|------|-----|------|
| Hold-out baseline (`autumn_2025`) | 940 | 1.644 | 0.931 | 16.2% |
| **Hold-out model** | **481** | **931** | **0.978** | **8.7%** |
| 8-fold CV mean (model) | 517±77 | 963±179 | 0.971 | — |
| 8-fold CV mean (baseline) | 825±115 | 1.540±225 | 0.926 | — |

- Cải thiện: **+48.9% MAE** (hold-out), **+37.4% MAE** (CV trung bình).
- Hyperparams chọn trên val: `RESID_CLIP=±0.55`, blend MLP weight **0.80**.
- Fold **winter** kém nhất (MAE ~674); spring/summer tốt nhất (~417–466).
- GTFS per-route operating hours cho `OPT_SCOPE`; embedding 25 routes trong NN (4 tuyến thiếu data sau lọc giờ).

## 5. Định nghĩa bài toán tối ưu hoá lịch trình

**Biến**: `trips[r,d,h]` trên `OPT_SCOPE` (giờ có lịch GTFS theo tuyến).

**Mục tiêu**: min Σ (D·30/trips + λ·trips); `LAMBDA_AUTO_CALIBRATE` + **dynamic TRIPS_MAX** (peak×1.15 / off-peak×1.35 / overnight×1.10).


In [ ]:
N_SLOTS = len(OPT_SCOPE)
print(f"Scope tối ưu: {len(OPT_ROUTES)} tuyến × {len(DIRECTIONS)} hướng × {len(OPT_HOURS)} giờ = {N_SLOTS} slots")

slot_to_idx = {s: i for i, s in enumerate(OPT_SCOPE)}
slot_route = np.array([s[0] for s in OPT_SCOPE])
slot_dir   = np.array([s[1] for s in OPT_SCOPE])
slot_hour  = np.array([s[2] for s in OPT_SCOPE])

hw_lookup = headway.set_index(["route_id", "direction_id", "hour"])["trip_count"]
hw_rd_median = headway.groupby(["route_id", "direction_id"])["trip_count"].median()
hw_rh_median = headway.groupby(["route_id", "hour"])["trip_count"].median()
hw_hour_median = headway.groupby("hour")["trip_count"].median()
hw_global_median = float(headway["trip_count"].median())

n_fallback = 0
fallback_tiers = []
baseline_trips = np.zeros(N_SLOTS)
for i, (r, d, h) in enumerate(OPT_SCOPE):
    if (r, d, h) in hw_lookup.index:
        baseline_trips[i] = hw_lookup.loc[(r, d, h)]
        fallback_tiers.append(0)
    elif (r, d) in hw_rd_median.index:
        baseline_trips[i] = hw_rd_median.loc[(r, d)]
        fallback_tiers.append(1)
        n_fallback += 1
    elif (r, h) in hw_rh_median.index:
        baseline_trips[i] = hw_rh_median.loc[(r, h)]
        fallback_tiers.append(2)
        n_fallback += 1
    elif h in hw_hour_median.index:
        baseline_trips[i] = hw_hour_median.loc[h]
        fallback_tiers.append(3)
        n_fallback += 1
    else:
        baseline_trips[i] = hw_global_median
        fallback_tiers.append(4)
        n_fallback += 1

print(f"Baseline trips: mean={baseline_trips.mean():.1f}, "
      f"min={baseline_trips.min():.0f}, max={baseline_trips.max():.0f}")
print(f"Headway fallback: {n_fallback}/{N_SLOTS} slots "
      f"(tier 0=exact GTFS, 1=route×dir, 2=route×hour, 3=hour, 4=global)")

TRIPS_MIN, TRIPS_MAX = srp.build_dynamic_bounds(
    baseline_trips,
    slot_hour,
    peak_factor=TRIPS_PEAK_MAX_FACTOR,
    offpeak_factor=TRIPS_OFFPEAK_MAX_FACTOR,
    overnight_factor=TRIPS_OVERNIGHT_MAX_FACTOR,
    min_factor=TRIPS_MIN_FACTOR,
    max_delta=TRIPS_MAX_DELTA,
)
print(f"λ mặc định LAMBDA_COST = {LAMBDA_COST}")
print(
    f"TRIPS bounds (dynamic): min×{TRIPS_MIN_FACTOR} | "
    f"max peak×{TRIPS_PEAK_MAX_FACTOR} offpeak×{TRIPS_OFFPEAK_MAX_FACTOR} "
    f"overnight×{TRIPS_OVERNIGHT_MAX_FACTOR} | delta=+{TRIPS_MAX_DELTA}"
)
print(f"TRIPS_MIN range: [{TRIPS_MIN.min()}, {TRIPS_MIN.max()}]")
print(f"TRIPS_MAX range: [{TRIPS_MAX.min()}, {TRIPS_MAX.max()}]")

print("Baseline bound theo giờ (peak / overnight / off_peak) — toàn mạng:")
_bh0 = srp.report_bound_status(
    baseline_trips, TRIPS_MIN, TRIPS_MAX, slot_hour, verbose=False
)["by_hour_group"]
for _, row in _bh0.iterrows():
    print(f"  {row['hour_group']:10s} n={int(row['n_slots']):4d} | "
          f"min={row['at_min_pct']:.0f}% max={row['at_max_pct']:.0f}% interior={row['interior_pct']:.0f}%")
print(f"Baseline total trips/day: {baseline_trips.sum():.0f} | max allowed: {TRIPS_MAX.sum():.0f} "
      f"(+{(TRIPS_MAX.sum()/baseline_trips.sum()-1)*100:.0f}%)")
# Tỷ lệ demand theo chiều (proxy từ cơ cấu chuyến GTFS baseline)
_dir_df = pd.DataFrame({
    "route_id": slot_route,
    "direction_id": slot_dir,
    "hour": slot_hour,
    "base_trips": baseline_trips,
})
_dir_sum = _dir_df.groupby(["route_id", "hour"])["base_trips"].transform("sum").clip(lower=1e-6)
direction_share = (_dir_df["base_trips"] / _dir_sum).to_numpy(dtype=float)
print(
    f"Direction share (route×hour): min={direction_share.min():.2f}, max={direction_share.max():.2f}, "
    f"mean={direction_share.mean():.2f}"
)


In [ ]:
def _scenario_context(scenario: str) -> tuple[int, int, float, int, int, pd.Series]:
    """Trả về (dow, is_weekend, rain_mm, is_rain, is_holiday, median factors)."""
    if scenario == "weekday_peak":
        dow, is_wkd = 1, 0
        rain_mm, is_rain, is_holiday = 0.0, 0, 0
        med = df.median(numeric_only=True)
    elif scenario == "weekend":
        dow, is_wkd = 5, 1
        rain_mm, is_rain, is_holiday = 0.0, 0, 0
        med = df.median(numeric_only=True)
    elif scenario == "rainy_day":
        dow, is_wkd = 1, 0
        is_holiday = 0
        rain_mask = pd.Series(False, index=df.index)
        if "is_rain" in df.columns:
            rain_mask = rain_mask | (df["is_rain"].fillna(0).astype(int) == 1)
        if "rain_mm" in df.columns:
            rain_mask = rain_mask | (pd.to_numeric(df["rain_mm"], errors="coerce").fillna(0) > 0)
        rain_ref = df[rain_mask] if rain_mask.any() else df
        rain_mm = float(pd.to_numeric(rain_ref.get("rain_mm", 0), errors="coerce").median())
        if not np.isfinite(rain_mm):
            rain_mm = 12.0
        is_rain = 1
        med = rain_ref.median(numeric_only=True)
    else:
        raise ValueError(scenario)
    return dow, is_wkd, rain_mm, is_rain, is_holiday, med


def build_scenario_features(scenario: str) -> pd.DataFrame:
    """Feature frame theo route×hour (một dòng / giờ / tuyến — không lặp chiều)."""
    dow, is_wkd, rain_mm, is_rain, is_holiday, med = _scenario_context(scenario)
    seen: set[tuple[str, int]] = set()
    rows = []
    for (r, _d, h) in OPT_SCOPE:
        key = (r, h)
        if key in seen:
            continue
        seen.add(key)
        row = dict(
            route_id=r, hour=h, day_of_week=dow,
            is_weekend=is_wkd, is_us_holiday=is_holiday,
            hour_sin=np.sin(2 * np.pi * h / 24), hour_cos=np.cos(2 * np.pi * h / 24),
            dow_sin=np.sin(2 * np.pi * dow / 7), dow_cos=np.cos(2 * np.pi * dow / 7),
        )
        row["month"] = int(med.get("month", 6))
        row.update(
            temperature_c=float(med.get("temperature_c", 5.0)),
            apparent_temperature_c=float(med.get("apparent_temperature_c", 3.0)),
            precipitation_mm=float(med.get("precipitation_mm", rain_mm)) if is_rain else rain_mm,
            rain_mm=rain_mm,
            snowfall_cm=float(med.get("snowfall_cm", 0.0)),
            windspeed_kmh=float(med.get("windspeed_kmh", 15.0)),
            windgusts_kmh=float(med.get("windgusts_kmh", 25.0)),
            is_rain=is_rain, is_snow=int(med.get("is_snow", 0) > 0),
            is_severe_wind=int(med.get("is_severe_wind", 0) > 0),
            is_peak_morning=int(h in (7, 8, 9)),
            is_peak_evening=int(h in (17, 18, 19)),
            is_overnight=int(h <= 5 or h >= 23),
            is_major_event_window=int(med.get("is_major_event_window", 0) > 0),
        )
        if USE_LAG_FEATURES:
            defaults = lag_feature_defaults.get(f"{r}|{h}", {})
            row.update(
                log_lag_24h=float(defaults.get("log_lag_24h", 0.0)),
                log_lag_168h=float(defaults.get("log_lag_168h", 0.0)),
                log_rolling_7d=float(defaults.get("log_rolling_7d", 0.0)),
            )
        rows.append(row)
    feat = pd.DataFrame(rows)
    feat = srp.add_cyclical_time_features(feat)
    feat = srp.add_weather_interaction_features(feat)
    feat["route_idx"] = feat["route_id"].map(route_to_idx).astype(int)
    return feat


def _predict_route_hour_demand(feat_rh: pd.DataFrame) -> pd.DataFrame:
    """Dự báo demand route×hour; trả về cột route_id, hour, demand."""
    bl = feat_rh.merge(baseline_lookup, on=["route_id", "hour", "is_weekend"], how="left")
    bl = bl.merge(fallback, on=["route_id", "hour"], how="left")

    fb_route = (
        df.groupby(["route_id", "is_weekend"], as_index=False)["demand"]
        .median().rename(columns={"demand": "fb_route_weekend"})
    )
    fb_hour = (
        df.groupby(["hour", "is_weekend"], as_index=False)["demand"]
        .median().rename(columns={"demand": "fb_hour_weekend"})
    )
    fb_global = float(pd.to_numeric(df["demand"], errors="coerce").dropna().median())
    if not np.isfinite(fb_global) or fb_global <= 0:
        fb_global = 100.0

    bl = bl.merge(fb_route, on=["route_id", "is_weekend"], how="left")
    bl = bl.merge(fb_hour, on=["hour", "is_weekend"], how="left")
    bl["baseline_demand"] = (
        pd.to_numeric(bl["baseline_demand"], errors="coerce")
        .fillna(bl["fallback_demand"])
        .fillna(bl["fb_route_weekend"])
        .fillna(bl["fb_hour_weekend"])
        .fillna(fb_global)
        .clip(lower=1e-6)
    )

    for c in NUM_FEATURES:
        if c == "log_baseline":
            continue
        if c not in bl.columns:
            bl[c] = 0.0
        bl[c] = pd.to_numeric(bl[c], errors="coerce")
        med = float(pd.to_numeric(df[c], errors="coerce").median()) if c in df.columns else 0.0
        bl[c] = bl[c].fillna(med if np.isfinite(med) else 0.0)

    bl["log_baseline"] = np.log1p(bl["baseline_demand"])
    Xn = scaler.transform(bl[NUM_FEATURES])
    pred = predict_demand(
        bl["route_idx"].values.reshape(-1, 1), Xn, bl["log_baseline"].values
    )
    fill = float(np.nanmedian(bl["baseline_demand"]))
    pred = np.clip(np.nan_to_num(pred, nan=fill), 0.0, None)
    return bl[["route_id", "hour"]].assign(demand=pred)


def allocate_demand_to_slots(route_hour: pd.DataFrame) -> np.ndarray:
    """Map demand route×hour → slot OPT_SCOPE; chia theo direction_share nếu bật."""
    rh = route_hour.set_index(["route_id", "hour"])["demand"]
    out = np.zeros(N_SLOTS, dtype=float)
    for i, (r, _d, h) in enumerate(OPT_SCOPE):
        d_rh = float(rh.get((r, h), np.nan))
        if not np.isfinite(d_rh):
            d_rh = float(rh.mean()) if len(rh) else 100.0
        if SPLIT_DEMAND_BY_DIRECTION:
            out[i] = d_rh * direction_share[i]
        else:
            out[i] = d_rh / max(len(DIRECTIONS), 1)
    return out


def apply_spillover_to_route_hour(
    route_hour: pd.DataFrame,
    *,
    alpha: float | None = None,
) -> pd.DataFrame:
    """BƯỚC 2 — spillover trên route×hour trước khi allocate slot."""
    a = SPILLOVER_ALPHA if alpha is None else float(alpha)
    if not USE_DEMAND_SPILLOVER or a <= 0:
        return route_hour
    hours_sorted = sorted(route_hour["hour"].unique())
    pred_dict = srp.route_hour_df_to_demand_dict(
        route_hour, OPT_ROUTES, hours_sorted, demand_col="demand"
    )
    adj_dict = srp.adjust_demand_with_spillover(pred_dict, transfer_matrix, alpha=a)
    return srp.demand_dict_to_route_hour_df(adj_dict, hours_sorted)


def get_scenario_demand(scenario: str, *, spillover_alpha: float | None = None) -> np.ndarray:
    feat_rh = build_scenario_features(scenario)
    route_hour = _predict_route_hour_demand(feat_rh)
    route_hour = apply_spillover_to_route_hour(route_hour, alpha=spillover_alpha)
    return allocate_demand_to_slots(route_hour)


SCENARIOS = ["weekday_peak", "weekend", "rainy_day"]
scenario_demand = {s: get_scenario_demand(s) for s in SCENARIOS}

_chk = build_scenario_features("rainy_day").iloc[0]
assert _chk["is_rain"] == 1
print("✓ rainy_day: is_rain=1, rain_mm=", float(_scenario_context("rainy_day")[2]))

scen_df = pd.DataFrame({s: scenario_demand[s] for s in SCENARIOS})
scen_df.insert(0, "hour", slot_hour)
scen_df.insert(0, "dir", slot_dir)
scen_df.insert(0, "route", slot_route)

# Kiểm tra demand khác nhau giữa 2 chiều (cùng giờ)
_dir_cmp = scen_df.groupby(["hour", "dir"])["weekday_peak"].mean().unstack()
if _dir_cmp.shape[1] >= 2:
    ratio = (_dir_cmp.iloc[:, 1] / _dir_cmp.iloc[:, 0].clip(lower=1)).replace([np.inf], np.nan)
    print(f"Demand dir1/dir0 (weekday_peak, TB): min={ratio.min():.2f}, max={ratio.max():.2f}")

hour_agg = scen_df.groupby("hour")[SCENARIOS].mean().round(0)
print("\nDemand TB theo giờ (passengers/hour/slot, 3 kịch bản):")
print(hour_agg.head(8), "\n...", hour_agg.tail(3))
_ratio = scenario_demand["rainy_day"] / np.maximum(scenario_demand["weekday_peak"], 1e-9)
print("\nRainy / weekday_peak (TB):", round(float(_ratio.mean()), 3))



In [ ]:
def evaluate_schedule(
    trips: np.ndarray,
    demand: np.ndarray,
    lambda_cost: float | None = None,
) -> dict:
    """Tính chỉ số lịch trình; lambda_cost=None → dùng LAMBDA_COST global."""
    lam = LAMBDA_COST if lambda_cost is None else float(lambda_cost)

    trips = np.asarray(trips, dtype=float)
    demand = np.asarray(demand, dtype=float)
    trips = np.nan_to_num(trips, nan=1.0, posinf=60.0, neginf=1.0)
    demand = np.nan_to_num(demand, nan=0.0, posinf=0.0, neginf=0.0)
    trips = np.maximum(trips, 1)
    demand = np.clip(demand, 0.0, None)

    headway_min = 60.0 / trips
    avg_wait_min = headway_min / 2.0
    passenger_min_wait = demand * avg_wait_min
    total_wait = passenger_min_wait.sum()
    weighted_avg_wait = total_wait / max(demand.sum(), 1e-9)
    total_fleet_cost = lam * trips.sum()
    return dict(
        total_passenger_min_wait=float(total_wait),
        weighted_avg_wait_min=float(weighted_avg_wait),
        total_fleet_cost=float(total_fleet_cost),
        total_trips=float(trips.sum()),
        headway_std=float(headway_min.std()),
        objective=float(total_wait + total_fleet_cost),
        lambda_cost=lam,
    )


CAPACITY_PER_TRIP = 1200


def evaluate_schedule_v2(
    trips: np.ndarray,
    demand: np.ndarray,
    lambda_cost: float | None = None,
    *,
    capacity_per_trip: float = CAPACITY_PER_TRIP,
) -> dict:
    """evaluate_schedule với wait metric overflow (peak crowding)."""
    lam = LAMBDA_COST if lambda_cost is None else float(lambda_cost)
    trips = np.asarray(trips, dtype=float)
    demand = np.asarray(demand, dtype=float)
    trips = np.nan_to_num(trips, nan=1.0, posinf=60.0, neginf=1.0)
    demand = np.nan_to_num(demand, nan=0.0, posinf=0.0, neginf=0.0)
    trips = np.maximum(trips, 1)
    demand = np.clip(demand, 0.0, None)

    m = srp.compute_wait_with_overflow(
        demand,
        trips,
        slot_route=slot_route,
        slot_dir=slot_dir,
        slot_hour=slot_hour,
        capacity_per_trip=capacity_per_trip,
        lambda_cost=lam,
    )
    headway_min = 60.0 / trips
    return dict(
        total_passenger_min_wait=float(m["total_passenger_min_wait"]),
        weighted_avg_wait_min=float(m["weighted_avg_wait_min"]),
        total_fleet_cost=float(m["total_fleet_cost"]),
        total_trips=float(m["total_trips"]),
        headway_std=float(headway_min.std()),
        objective=float(m["objective"]),
        lambda_cost=lam,
        overflow_pct=float(m["overflow_pct"]),
        total_overflow_pax=float(m["total_overflow_pax"]),
        slot_wait=m["slot_wait"],
        overflow_out=m["overflow_out"],
    )


def objective(trips: np.ndarray, demand: np.ndarray, lambda_cost: float | None = None) -> float:
    return evaluate_schedule(trips, demand, lambda_cost=lambda_cost)["objective"]

def optimize_schedule_analytical(
    demand: np.ndarray,
    lambda_cost: float | None = None,
    trips_min: np.ndarray | None = None,
    trips_max: np.ndarray | None = None,
) -> np.ndarray:
    """Per-slot optimum: min D·30/t + λ·t  →  t* = sqrt(30·D/λ), clip bounds."""
    lam = LAMBDA_COST if lambda_cost is None else float(lambda_cost)
    tmin = np.asarray(trips_min if trips_min is not None else TRIPS_MIN, dtype=float)
    tmax = np.asarray(trips_max if trips_max is not None else TRIPS_MAX, dtype=float)
    trips_star = np.sqrt(30.0 * np.maximum(demand, 1e-9) / lam)
    return np.clip(np.round(trips_star), tmin, tmax).astype(int)


def summarize_bound_status(trips: np.ndarray, label: str = "") -> dict:
    return srp.report_bound_status(
        trips, TRIPS_MIN, TRIPS_MAX, slot_hour, label=label, verbose=bool(label)
    )


def calibrate_lambda_for_demand(
    demand: np.ndarray,
    lambda_lo: float = 80.0,
    lambda_hi: float = 2500.0,
    target_max_frac: float = 0.25,
) -> float:
    lo, hi = float(lambda_lo), float(lambda_hi)
    for _ in range(48):
        mid = (lo + hi) / 2.0
        sol = optimize_schedule_analytical(demand, lambda_cost=mid)
        at_max = float((sol >= TRIPS_MAX).mean())
        if at_max > target_max_frac:
            lo = mid
        else:
            hi = mid
    return hi


def resolve_lambda_for_demand(
    demand: np.ndarray,
    lambda_cost: float,
    *,
    auto_calibrate: bool | None = None,
    target_max_frac: float | None = None,
) -> tuple[float, np.ndarray]:
    auto = LAMBDA_AUTO_CALIBRATE if auto_calibrate is None else auto_calibrate
    target = TARGET_MAX_BIND_FRACTION if target_max_frac is None else target_max_frac
    lam = float(lambda_cost)
    sol = optimize_schedule_analytical(demand, lambda_cost=lam)
    if auto and float((sol >= TRIPS_MAX).mean()) > target:
        lam = calibrate_lambda_for_demand(demand, lambda_lo=lam, target_max_frac=target)
        sol = optimize_schedule_analytical(demand, lambda_cost=lam)
    return lam, sol


baseline_metrics = {s: evaluate_schedule(baseline_trips, scenario_demand[s])
                    for s in scenario_demand}
print("Baseline metrics (λ={}):".format(LAMBDA_COST))
for s, m in baseline_metrics.items():
    print(f"  {s:15s} | wait={m['total_passenger_min_wait']:>14,.0f} | "
          f"fleet={m['total_fleet_cost']:>12,.0f} | obj={m['objective']:,.0f} | trips={m['total_trips']:.0f}")


def pct_improve(base_val: float, new_val: float) -> float:
    """% cải thiện (base - new) / base, có guard chia 0."""
    base = float(base_val)
    return (base - float(new_val)) / max(base, 1e-9) * 100.0


def resolve_opt_policy(
    target: str | None = None,
    weights: dict | None = None,
    lambda_candidates: list[float] | None = None,
) -> tuple[str, tuple[float, float], list[float]]:
    """Chuẩn hóa policy tối ưu để dùng lại giữa các block."""
    opt_target = str(target if target is not None else globals().get("OPT_TARGET", "objective")).strip().lower()
    if opt_target not in {"wait", "cost", "objective", "balanced"}:
        opt_target = "objective"

    cfg = weights if weights is not None else globals().get("BALANCED_WEIGHTS", {"wait": 0.5, "cost": 0.5})
    w_wait = max(float(cfg.get("wait", 0.5)), 0.0)
    w_cost = max(float(cfg.get("cost", 0.5)), 0.0)
    if (w_wait + w_cost) <= 1e-9:
        w_wait, w_cost = 0.5, 0.5
    w_sum = w_wait + w_cost
    w_wait, w_cost = w_wait / w_sum, w_cost / w_sum

    cands = lambda_candidates if lambda_candidates is not None else globals().get("LAMBDA_CANDIDATES", [LAMBDA_COST])
    cands = [float(x) for x in cands]
    if not cands:
        cands = [float(LAMBDA_COST)]

    return opt_target, (w_wait, w_cost), cands


def score_metrics(metrics: dict, base_metrics: dict, opt_target: str, weights: tuple[float, float]) -> float:
    """Tính score theo target policy."""
    if opt_target == "wait":
        return float(metrics["total_passenger_min_wait"])
    if opt_target == "cost":
        return float(metrics["total_fleet_cost"])
    if opt_target == "balanced":
        w_wait, w_cost = weights
        wait_ratio = float(metrics["total_passenger_min_wait"]) / max(float(base_metrics["total_passenger_min_wait"]), 1e-9)
        cost_ratio = float(metrics["total_fleet_cost"]) / max(float(base_metrics["total_fleet_cost"]), 1e-9)
        return w_wait * wait_ratio + w_cost * cost_ratio
    return float(metrics["objective"])


def select_best_lambda_solution(
    demand: np.ndarray,
    baseline_trips_arr: np.ndarray,
    lambda_eval: float,
    opt_target: str,
    weights: tuple[float, float],
    lambda_candidates: list[float],
) -> tuple[np.ndarray, float, dict, dict, float]:
    """Quét lambda_opt và chọn nghiệm tốt nhất theo policy."""
    base_metrics = evaluate_schedule(baseline_trips_arr, demand, lambda_cost=lambda_eval)

    best_score = float("inf")
    best_sol = None
    best_lam = float(lambda_eval)
    best_eval = None

    for lam_opt in lambda_candidates:
        cand = optimize_schedule_analytical(demand, lambda_cost=lam_opt)
        cand = np.clip(cand, TRIPS_MIN, TRIPS_MAX).astype(int)
        m_eval = evaluate_schedule(cand, demand, lambda_cost=lambda_eval)
        score = score_metrics(m_eval, base_metrics, opt_target, weights)
        if score + 1e-9 < best_score:
            best_score = score
            best_sol = cand.copy()
            best_lam = float(lam_opt)
            best_eval = m_eval

    return best_sol, best_lam, best_eval, base_metrics, best_score


## 6. Tối ưu lịch trình (Analytical + ràng buộc GTFS)

**Khung phục vụ** theo `(route, direction)`: đọc **first/last departure** từ GTFS → `service_windows.csv`.

- `OPT_SCOPE` dùng giờ theo từng chiều (không gộp 2 hướng).
- Sau analytical: `apply_service_window_constraints` — giữ trips tại giờ đầu/cuối ≥ baseline; ngoài khung không tăng.

**Mặc định** analytical (`USE_ANALYTICAL_OPT=True`). GA+Tabu (`RUN_GA_TABU=True`) chỉ chạy khi tắt analytical **và** `N_SLOTS ≤ GA_MAX_SLOTS` (full run: **bỏ qua**, N_SLOTS=1.182).

**Kết quả analytical (run hiện tại, `OPT_TARGET=balanced`):**

| Kịch bản | λ_opt | Trips | Chờ TB (phút) | Objective impr. @λ_opt | Bound TRIPS_MAX |
|----------|-------|-------|---------------|------------------------|-----------------|
| weekday_peak | 260 | 8.497→10.465 | 3.86→2.91 | **−14.4%** | **58%** |
| weekend | 260 | →10.351 | 4.05→3.04 | −11.1% | 58% |
| rainy_day | 260 | →10.462 | 3.86→2.91 | −14.3% | 58% |

- Với bound uniform ×1.25: ~58% slot ở TRIPS_MAX (ép trần). Dynamic bound (peak×1.15 / off-peak×1.35) giảm bind peak, tăng interior off-peak.
- Baseline: 8.497 trips/ngày; trần cho phép: 12.091 (+42%).


In [ ]:
def clip_trips(individual: np.ndarray) -> np.ndarray:
    return np.clip(individual, TRIPS_MIN, TRIPS_MAX).astype(int)

def init_population(pop_size: int, baseline: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    pop = np.zeros((pop_size, N_SLOTS), dtype=int)
    n_seed = int(pop_size * 0.75)
    for i in range(n_seed):
        pop[i] = clip_trips(baseline + rng.normal(0, 3, N_SLOTS))
    for i in range(n_seed, pop_size):
        pop[i] = rng.integers(TRIPS_MIN, TRIPS_MAX + 1)
    return pop

def tournament_select(pop, fitness, k, rng):
    idxs = rng.integers(0, len(pop), size=k)
    return pop[idxs[np.argmin(fitness[idxs])]].copy()

def uniform_crossover(p1, p2, rng):
    mask = rng.random(N_SLOTS) < 0.5
    return np.where(mask, p1, p2), np.where(mask, p2, p1)

def mutate(ind, p_mut, sigma, rng):
    mask = rng.random(N_SLOTS) < p_mut
    return clip_trips(ind + rng.normal(0, sigma, N_SLOTS) * mask)

def genetic_algorithm(
    demand: np.ndarray,
    pop_size: int = 80,
    generations: int = 120,
    p_cross: float = 0.85,
    p_mut: float = 0.15,
    sigma: float = 3.0,
    elitism: int = 2,
    seed: int = SEED,
    lambda_cost: float | None = None,
) -> tuple[np.ndarray, dict]:
    lam = LAMBDA_COST if lambda_cost is None else float(lambda_cost)
    rng = np.random.default_rng(seed)
    pop = init_population(pop_size, baseline_trips, rng)
    fitness = np.array([objective(ind, demand, lambda_cost=lam) for ind in pop])
    log_best, log_avg = [], []
    best_ind = pop[np.argmin(fitness)].copy()
    best_fit = float(fitness.min())
    for _ in range(generations):
        elite_idx = np.argsort(fitness)[:elitism]
        new_pop = [pop[i].copy() for i in elite_idx]
        while len(new_pop) < pop_size:
            p1 = tournament_select(pop, fitness, 3, rng)
            p2 = tournament_select(pop, fitness, 3, rng)
            if rng.random() < p_cross:
                c1, c2 = uniform_crossover(p1, p2, rng)
            else:
                c1, c2 = p1.copy(), p2.copy()
            new_pop.extend([mutate(c1, p_mut, sigma, rng), mutate(c2, p_mut, sigma, rng)])
        pop = np.array(new_pop[:pop_size])
        fitness = np.array([objective(ind, demand, lambda_cost=lam) for ind in pop])
        gen_best = float(fitness.min())
        if gen_best < best_fit:
            best_fit = gen_best
            best_ind = pop[np.argmin(fitness)].copy()
        log_best.append(best_fit)
        log_avg.append(float(fitness.mean()))
    return best_ind, dict(best_log=log_best, avg_log=log_avg, lambda_cost=lam)

def tabu_search(
    start: np.ndarray,
    demand: np.ndarray,
    *,
    iters: int = 400,
    tabu_size: int = 24,
    lambda_cost: float | None = None,
) -> tuple[np.ndarray, dict]:
    lam = LAMBDA_COST if lambda_cost is None else float(lambda_cost)
    current = clip_trips(start.copy())
    best = current.copy()
    best_fit = objective(best, demand, lambda_cost=lam)
    tabu: list[tuple[int, int]] = []
    log_best = []
    for _ in range(iters):
        best_move = None
        best_neighbor = None
        best_neighbor_fit = float("inf")
        for slot in range(N_SLOTS):
            for delta in (-1, 1):
                cand = current.copy()
                nv = int(cand[slot]) + delta
                if nv < TRIPS_MIN[slot] or nv > TRIPS_MAX[slot]:
                    continue
                cand[slot] = nv
                move = (slot, delta)
                fit = objective(cand, demand, lambda_cost=lam)
                if move in tabu and fit >= best_fit:
                    continue
                if fit < best_neighbor_fit:
                    best_neighbor_fit = fit
                    best_neighbor = cand
                    best_move = move
        if best_neighbor is None or best_move is None:
            break
        current = best_neighbor
        if best_neighbor_fit < best_fit:
            best = current.copy()
            best_fit = best_neighbor_fit
        tabu.append(best_move)
        if len(tabu) > tabu_size:
            tabu.pop(0)
        log_best.append(best_fit)
    return best, dict(best_log=log_best, lambda_cost=lam)


In [ ]:
service_windows = srp.build_route_direction_departure_windows(
    SCHEDULE_DIR, service_id=HEADWAY_SERVICE
)
service_windows.to_csv(OUT_DIR / "service_windows.csv", index=False)
print(f"GTFS service windows: {len(service_windows)} (route×direction)")
print(service_windows.head(8).to_string(index=False))

def optimize_schedule_constrained(
    demand: np.ndarray,
    lambda_cost: float | None = None,
) -> np.ndarray:
    """Analytical per-slot + ràng buộc first/last departure GTFS."""
    raw = optimize_schedule_analytical(demand, lambda_cost=lambda_cost)
    return srp.apply_service_window_constraints(
        raw,
        slot_route=slot_route,
        slot_dir=slot_dir,
        slot_hour=slot_hour,
        windows=service_windows,
        baseline_trips=baseline_trips,
        trips_min=TRIPS_MIN,
        trips_max=TRIPS_MAX,
    )


def resolve_lambda_for_demand_constrained(
    demand: np.ndarray,
    lambda_cost: float,
    **kwargs,
) -> tuple[float, np.ndarray]:
    """Calibrate λ rồi tối ưu có ràng buộc GTFS."""
    auto = kwargs.get("auto_calibrate", LAMBDA_AUTO_CALIBRATE)
    target = kwargs.get("target_max_frac", TARGET_MAX_BIND_FRACTION)
    lam = float(lambda_cost)
    sol = optimize_schedule_constrained(demand, lambda_cost=lam)
    if auto and float((sol >= TRIPS_MAX).mean()) > target:
        lo, hi = lam, 2500.0
        for _ in range(48):
            mid = (lo + hi) / 2.0
            sol_mid = optimize_schedule_constrained(demand, lambda_cost=mid)
            if float((sol_mid >= TRIPS_MAX).mean()) > target:
                lo = mid
            else:
                hi = mid
        lam = hi
        sol = optimize_schedule_constrained(demand, lambda_cost=lam)
    return lam, sol


In [ ]:
opt_results = {}
lambda_used = {}
bound_status = {}
bound_by_hour = {}
ga_results = {}
ga_logs = {}
tabu_results = {}
tabu_logs = {}
import time

OPT_METHOD = "Analytical" if USE_ANALYTICAL_OPT else "GA+Tabu"
print(f"Optimizer: {OPT_METHOD} | N_SLOTS={N_SLOTS}")

for scen, dem in scenario_demand.items():
    t0 = time.time()
    if USE_ANALYTICAL_OPT:
        opt_target, weights, lam_cands = resolve_opt_policy()
        if opt_target in ("wait", "cost", "balanced") and len(lam_cands) > 1:
            best_sol, lam, m_eval, _, _ = select_best_lambda_solution(
                dem, baseline_trips, LAMBDA_COST, opt_target, weights, lam_cands
            )
            best_sol = srp.apply_service_window_constraints(
                best_sol,
                slot_route=slot_route, slot_dir=slot_dir, slot_hour=slot_hour,
                windows=service_windows,
                baseline_trips=baseline_trips, trips_min=TRIPS_MIN, trips_max=TRIPS_MAX,
            )
            opt_results[scen] = best_sol
        else:
            lam, opt_results[scen] = resolve_lambda_for_demand_constrained(dem, LAMBDA_COST)
        bound_status[scen] = summarize_bound_status(
            opt_results[scen], label=f"{scen} final λ={lam:.0f}"
        )
        bound_by_hour[scen] = bound_status[scen]["by_hour_group"].to_dict(orient="records")
    else:
        lam = float(LAMBDA_COST)
        opt_results[scen], _ = genetic_algorithm(
            dem, generations=GA_GENERATIONS, pop_size=GA_POP_SIZE, seed=SEED
        )
        opt_results[scen] = srp.apply_service_window_constraints(
            opt_results[scen],
            slot_route=slot_route, slot_dir=slot_dir, slot_hour=slot_hour,
            windows=service_windows,
            baseline_trips=baseline_trips, trips_min=TRIPS_MIN, trips_max=TRIPS_MAX,
        )
    lambda_used[scen] = lam
    m = evaluate_schedule(opt_results[scen], dem, lambda_cost=lam)
    base_obj = baseline_metrics[scen]["objective"]
    impr = (base_obj - m["objective"]) / base_obj * 100
    label = "Analytical" if USE_ANALYTICAL_OPT else "GA"
    print(f"[{scen:14s}] {label} obj={m['objective']:>14,.0f} (impr {impr:+.2f}%) "
          f"trips={m['total_trips']:.0f} λ={lam:.0f} [{time.time()-t0:.2f}s]")

_sched = pd.DataFrame({
    "route": slot_route, "direction": slot_dir, "hour": slot_hour,
    "baseline_trips": baseline_trips,
    "opt_trips": opt_results["weekday_peak"],
    "demand_pred": scenario_demand["weekday_peak"],
})
_sched = _sched.merge(
    service_windows.rename(columns={"route_id": "route", "direction_id": "direction"}),
    on=["route", "direction"], how="left",
)
_sched.to_csv(OUT_DIR / "schedule_weekday_peak.csv", index=False)
print("Đã lưu:", OUT_DIR / "schedule_weekday_peak.csv", "|", OUT_DIR / "service_windows.csv")


## 7. Genetic Algorithm *(tùy chọn — không chạy mặc định)*

> `RUN_GA_TABU=True` và `USE_ANALYTICAL_OPT=False` và `N_SLOTS ≤ GA_MAX_SLOTS` (200). Full 29 routes: analytical, GA/Tabu tự bỏ qua.

**Run hiện tại:** `Bỏ qua GA — RUN_GA_TABU=True, analytical=True, N_SLOTS=1182 (max 200).`

Encoding: vector `trips[N_SLOTS]`; tournament + uniform crossover + mutation; elitism=2.


In [ ]:
# OPTIONAL — không chạy khi USE_ANALYTICAL_OPT=True (full 29 routes)
if RUN_GA_TABU and not USE_ANALYTICAL_OPT and N_SLOTS <= GA_MAX_SLOTS:
    ga_results = {}
    ga_logs = {}
    for scen, dem in scenario_demand.items():
        t0 = time.time()
        sol, log = genetic_algorithm(dem, generations=GA_GENERATIONS, pop_size=GA_POP_SIZE, seed=SEED)
        ga_results[scen] = sol
        ga_logs[scen] = log
        m = evaluate_schedule(sol, dem)
        impr = (baseline_metrics[scen]["objective"] - m["objective"]) / baseline_metrics[scen]["objective"] * 100
        print(f"[{scen}] GA impr {impr:+.2f}% [{time.time()-t0:.1f}s]")
else:
    print(f"Bỏ qua GA — RUN_GA_TABU={RUN_GA_TABU}, analytical={USE_ANALYTICAL_OPT}, N_SLOTS={N_SLOTS} (max {GA_MAX_SLOTS}).")


In [ ]:
if RUN_GA_TABU and not USE_ANALYTICAL_OPT and N_SLOTS <= GA_MAX_SLOTS and ga_logs:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, (scen, log) in zip(axes, ga_logs.items()):
        ax.plot(log["best_log"], label="Best", lw=2)
        ax.plot(log["avg_log"], label="Mean", alpha=0.5)
        ax.axhline(baseline_metrics[scen]["objective"], color="red", ls="--", label="Baseline")
        ax.set_title(f"GA — {scen}")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_ga_convergence.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print(f"Bỏ qua biểu đồ GA — RUN_GA_TABU={RUN_GA_TABU}, N_SLOTS={N_SLOTS}.")


## 8. Tabu Search *(tùy chọn — sau GA, không chạy mặc định)*

Neighborhood ±1 trip/slot; tabu_size=24; aspiration khi vượt best_global.

**Run hiện tại:** bỏ qua (cùng điều kiện §7 — cần GA output và N_SLOTS≤200).


In [ ]:
# OPTIONAL — Tabu refine sau GA
if RUN_GA_TABU and not USE_ANALYTICAL_OPT and N_SLOTS <= GA_MAX_SLOTS and ga_results:
    tabu_results = {}
    tabu_logs = {}
    for scen, dem in scenario_demand.items():
        t0 = time.time()
        best, log = tabu_search(ga_results[scen], dem, iters=TABU_ITERS)
        tabu_results[scen] = srp.apply_service_window_constraints(
            best,
            slot_route=slot_route, slot_dir=slot_dir, slot_hour=slot_hour,
            windows=service_windows,
            baseline_trips=baseline_trips, trips_min=TRIPS_MIN, trips_max=TRIPS_MAX,
        )
        tabu_logs[scen] = log
        m = evaluate_schedule(tabu_results[scen], dem)
        impr = (baseline_metrics[scen]["objective"] - m["objective"]) / baseline_metrics[scen]["objective"] * 100
        print(f"[{scen}] Tabu impr {impr:+.2f}% [{time.time()-t0:.1f}s]")
    opt_results = tabu_results
    OPT_METHOD = "GA+Tabu"
else:
    print(f"Bỏ qua Tabu — RUN_GA_TABU={RUN_GA_TABU}, analytical={USE_ANALYTICAL_OPT}, N_SLOTS={N_SLOTS}.")


## 8b. Wait metric v2 — overflow (peak crowding)

Metric cũ: `wait = headway/2` (Poisson). Metric mới: overflow khi `demand > trips × capacity` lan truyền theo chuỗi giờ cùng (route, direction); hành khách kẹt chờ `1.5 × headway`.

So sánh trên `weekday_peak`: baseline vs optimized; bar chart theo nhóm giờ; báo cáo % improvement bị **overestimate** bởi metric cũ.

In [ ]:
dem_peak = scenario_demand["weekday_peak"]
trips_opt = opt_results["weekday_peak"]

def _legacy_slot_wait(demand_arr, trips_arr):
    trips_arr = np.maximum(np.asarray(trips_arr, dtype=float), 1.0)
    return np.asarray(demand_arr, dtype=float) * (60.0 / trips_arr / 2.0)

def _compare_wait_metrics(trips_arr, label):
    m_v1 = evaluate_schedule(trips_arr, dem_peak, lambda_cost=LAMBDA_COST)
    m_v2 = evaluate_schedule_v2(trips_arr, dem_peak, lambda_cost=LAMBDA_COST)
    return dict(
        label=label,
        wait_v1=m_v1["total_passenger_min_wait"],
        wait_v2=m_v2["total_passenger_min_wait"],
        overflow_pct=m_v2["overflow_pct"],
        total_overflow_pax=m_v2["total_overflow_pax"],
        slot_wait_v1=_legacy_slot_wait(dem_peak, trips_arr),
        slot_wait_v2=m_v2["slot_wait"],
    )

cmp = {
    "baseline": _compare_wait_metrics(baseline_trips, "baseline"),
    "optimized": _compare_wait_metrics(trips_opt, "optimized"),
}

base = cmp["baseline"]
opt = cmp["optimized"]
impr_v1 = (base["wait_v1"] - opt["wait_v1"]) / max(base["wait_v1"], 1e-9) * 100
impr_v2 = (base["wait_v2"] - opt["wait_v2"]) / max(base["wait_v2"], 1e-9) * 100
overestimate_pp = impr_v1 - impr_v2
overestimate_rel = overestimate_pp / max(impr_v2, 1e-9) * 100

print("=== Wait metric: headway/2 (v1) vs overflow (v2) — weekday_peak ===")
for k, c in cmp.items():
    print(f"  [{c['label']:10s}] v1={c['wait_v1']:>14,.0f} pax-min | v2={c['wait_v2']:>14,.0f} pax-min | "
          f"overflow slots={c['overflow_pct']:.1f}% | overflow pax={c['total_overflow_pax']:,.0f}")
print(f"\nImprovement v1 (metric cũ):  {impr_v1:+.2f}%")
print(f"Improvement v2 (metric mới): {impr_v2:+.2f}%")
print(f"Overestimate (v1 − v2):      {overestimate_pp:+.2f} pp "
      f"({overestimate_rel:+.1f}% so với improvement thực v2)")

# Bar chart theo nhóm giờ
def _group_wait(slot_wait):
    return srp.wait_totals_by_hour_groups(slot_wait, slot_hour).set_index("hour_group")["total_wait"]

grp_order = ["peak", "off_peak", "overnight"]
v1_base = _group_wait(base["slot_wait_v1"]).reindex(grp_order, fill_value=0)
v1_opt = _group_wait(opt["slot_wait_v1"]).reindex(grp_order, fill_value=0)
v2_base = _group_wait(base["slot_wait_v2"]).reindex(grp_order, fill_value=0)
v2_opt = _group_wait(opt["slot_wait_v2"]).reindex(grp_order, fill_value=0)

x = np.arange(len(grp_order))
w = 0.2
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - 1.5 * w, v1_base, width=w, label="v1 baseline", color="#9ecae1")
ax.bar(x - 0.5 * w, v1_opt, width=w, label="v1 optimized", color="#3182bd")
ax.bar(x + 0.5 * w, v2_base, width=w, label="v2 baseline", color="#fdae6b")
ax.bar(x + 1.5 * w, v2_opt, width=w, label="v2 optimized", color="#e6550d")
ax.set_xticks(x)
ax.set_xticklabels(grp_order)
ax.set_ylabel("Passenger-min wait")
ax.set_title(f"Wait v1 vs v2 theo nhóm giờ (weekday_peak, cap={CAPACITY_PER_TRIP}/trip)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_wait_metric_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

wait_metric_cmp = pd.DataFrame([
    dict(schedule=k, wait_v1=c["wait_v1"], wait_v2=c["wait_v2"],
         overflow_pct=c["overflow_pct"], total_overflow_pax=c["total_overflow_pax"])
    for k, c in cmp.items()
])
wait_metric_cmp["improve_v1_pct"] = np.nan
wait_metric_cmp["improve_v2_pct"] = np.nan
wait_metric_cmp.loc[wait_metric_cmp["schedule"] == "optimized", "improve_v1_pct"] = impr_v1
wait_metric_cmp.loc[wait_metric_cmp["schedule"] == "optimized", "improve_v2_pct"] = impr_v2
wait_metric_cmp.to_csv(OUT_DIR / "wait_metric_comparison.csv", index=False)
print("Đã lưu:", OUT_DIR / "wait_metric_comparison.csv", "|", OUT_DIR / "fig_wait_metric_comparison.png")

## 9. Đánh giá & So sánh tổng hợp

So sánh **Baseline GTFS** vs **Analytical** (hoặc GA+Tabu nếu bật) trên 3 kịch bản. File chính:

- `results_summary.csv` — metric đầy đủ theo scenario/method.
- `tradeoff_summary.csv` — passenger-min wait saved vs fleet cost increase.
- `schedule_weekday_peak.csv` — trips baseline/opt theo slot.
- `baseline_metrics.json` — snapshot JSON (NN eval, λ sensitivity, routes).

**Kết quả full run (`default`, Analytical, λ_eval=150, λ_opt=260, 1.182 slots):**

| Kịch bản | Wait TB (phút) | Giảm chờ | Giảm objective | Trips | Fleet cost Δ |
|----------|----------------|----------|----------------|-------|--------------|
| weekday_peak | 3.86 → **2.91** | **−24.5%** | **−21.0%** | 8.497 → **10.465** | +295k |
| weekend | 4.05 → **3.04** | **−25.0%** | **−20.2%** | → **10.351** | +278k |
| rainy_day | 3.86 → **2.91** | **−24.5%** | **−21.0%** | → **10.462** | +295k |

- Passenger-min chờ tiết kiệm ~**3.9M** (weekday_peak) đổi lấy fleet cost +**~295k** (λ×trips) và **+23%** số chuyến.
- Objective @λ_eval=150 cao hơn log tối ưu @λ_opt=260 (~14% vs ~21%) — báo cáo thống nhất theo λ khi so sánh.
- GA/Tabu: bỏ qua đúng (`RUN_GA_TABU=True`, analytical=True, N_SLOTS>200).


In [ ]:
rows = []
for scen in scenario_demand:
    dem = scenario_demand[scen]
    base_wait = baseline_metrics[scen]["total_passenger_min_wait"]
    method_sols = [("Baseline", baseline_trips)]
    if USE_ANALYTICAL_OPT:
        method_sols.append(("Analytical", opt_results[scen]))
    elif ga_results and tabu_results:
        method_sols.extend([("GA", ga_results[scen]), ("GA+Tabu", tabu_results[scen])])
    else:
        method_sols.append((OPT_METHOD, opt_results[scen]))
    for method, sol in method_sols:
        m = evaluate_schedule(sol, dem)
        rows.append(dict(
            scenario=scen, method=method, n_slots=N_SLOTS,
            total_passenger_min_wait=m["total_passenger_min_wait"],
            total_fleet_cost=m["total_fleet_cost"],
            weighted_avg_wait_min=m["weighted_avg_wait_min"],
            headway_std=m["headway_std"],
            total_trips=m["total_trips"],
            objective=m["objective"],
            wait_saved_vs_baseline=base_wait - m["total_passenger_min_wait"],
            fleet_cost_delta_vs_baseline=m["total_fleet_cost"] - baseline_metrics[scen]["total_fleet_cost"],
        ))

results_df = pd.DataFrame(rows)
base_obj_per_scen = results_df[results_df["method"] == "Baseline"].set_index("scenario")["objective"]
base_wait_per_scen = results_df[results_df["method"] == "Baseline"].set_index("scenario")["total_passenger_min_wait"]
results_df["objective_improvement_pct"] = results_df.apply(
    lambda r: (base_obj_per_scen[r["scenario"]] - r["objective"]) / base_obj_per_scen[r["scenario"]] * 100, axis=1)
results_df["wait_improvement_pct"] = results_df.apply(
    lambda r: (base_wait_per_scen[r["scenario"]] - r["total_passenger_min_wait"]) / base_wait_per_scen[r["scenario"]] * 100, axis=1)

_lam_note = ", ".join(f"{s}={lambda_used.get(s, LAMBDA_COST):.0f}" for s in scenario_demand)
print(f"=== Kết quả ({OPT_METHOD}, λ_eval={LAMBDA_COST}, λ_opt: {_lam_note}, {N_SLOTS} slots) ===")
print(results_df.round(3).to_string(index=False))
results_df.to_csv(OUT_DIR / "results_summary.csv", index=False)

tradeoff_df = results_df[results_df["method"] == "Analytical"]
tradeoff_show = tradeoff_df[["scenario", "wait_saved_vs_baseline", "fleet_cost_delta_vs_baseline",
    "objective_improvement_pct", "total_trips"]].rename(columns={
    "wait_saved_vs_baseline": "passenger_min_wait_saved",
    "fleet_cost_delta_vs_baseline": "fleet_cost_increase",
})
print("\n=== Trade-off Analytical vs Baseline ===")
print(tradeoff_show.round(0).to_string(index=False))
tradeoff_show.to_csv(OUT_DIR / "tradeoff_summary.csv", index=False)

_bound_rows = [{"scenario": scen, **row} for scen, rows in bound_by_hour.items() for row in rows]
if _bound_rows:
    pd.DataFrame(_bound_rows).to_csv(OUT_DIR / "bound_by_hour.csv", index=False)
    print("Đã lưu:", OUT_DIR / "bound_by_hour.csv")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

best_label = "Analytical" if USE_ANALYTICAL_OPT else "GA+Tabu"
methods_impr = [best_label] if USE_ANALYTICAL_OPT else ["GA", "GA+Tabu"]
pivot = results_df.pivot(index="scenario", columns="method", values="wait_improvement_pct")
pivot = pivot[[c for c in methods_impr if c in pivot.columns]]
pivot.plot(kind="bar", ax=axes[0], color=["#2ECC71"] if USE_ANALYTICAL_OPT else ["#3498DB", "#2ECC71"])
axes[0].set_title("Cải thiện passenger-min chờ (%)")
axes[0].set_ylabel("Improvement (%)"); axes[0].legend(title="Method")

cols_wait = ["Baseline", best_label]
pivot2 = results_df.pivot(index="scenario", columns="method", values="weighted_avg_wait_min")[cols_wait]
pivot2.plot(kind="bar", ax=axes[1], color=["#95A5A6", "#2ECC71"])
axes[1].set_title("Thời gian chờ TB có trọng số (phút)")
axes[1].set_ylabel("Avg wait (min)"); axes[1].legend(title="Method")

plt.tight_layout()
plt.savefig(OUT_DIR / "fig_improvement.png", dpi=120, bbox_inches="tight")
plt.show()


## 9b. Trade-off chờ ↔ fleet, Pareto frontier & độ nhạy λ

- **Trade-off**: giảm `passenger-min wait` bằng cách tăng số chuyến → `fleet_cost = λ × trips` tăng. Run hiện tại: ~**−25%** chờ, **+23%** trips, fleet cost +**~23%**.
- **Pareto frontier** (`pareto_frontier.csv`, `fig_pareto_frontier.png`): Chebyshev scalarization với 20 trọng số `w∈[0.1,0.9]`; đánh dấu baseline, balanced (w≈0.5), knee (max curvature).
- **Độ nhạy λ** (`lambda_sensitivity.csv`): tái tối ưu **analytical** `weekday_peak` với λ ∈ `LAMBDA_GRID` (100…1000).

| λ_eval | λ_opt | Trips | % @ TRIPS_MAX | Cải thiện objective |
|--------|-------|-------|---------------|---------------------|
| 100 | 508 | 10.056 | 50% | **21.4%** |
| 150 | 508 | 10.056 | 50% | 20.5% |
| 200 | 508 | 10.056 | 50% | 19.5% |
| 400 | 508 | 10.056 | 50% | 16.2% |
| 600 | 600 | 9.917 | 48% | 13.5% |
| 1000 | 1000 | 9.364 | 34% | 9.7% |

- λ thấp (100–400): cùng lịch (λ_opt≈508), objective improvement cao hơn khi đánh giá với λ nhỏ.
- λ cao (600+): ít chuyến, ít chạm TRIPS_MAX hơn, trade-off nghiêng về fleet cost.
- Với bound uniform ×1.25, interior < 50%; dynamic bound (peak×1.15 / off-peak×1.35) giảm ép trần giờ cao điểm.

Biểu đồ: `fig_tradeoff.png`, `fig_pareto_frontier.png`, `fig_lambda_sensitivity.png`, `fig_improvement.png`.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
best_label = "Analytical" if USE_ANALYTICAL_OPT else "GA+Tabu"
for scen in SCENARIOS:
    sub = results_df[(results_df["scenario"] == scen) & (results_df["method"].isin(["Baseline", best_label]))]
    for method in ["Baseline", best_label]:
        r = sub[sub["method"] == method].iloc[0]
        ax.scatter(r["total_fleet_cost"], r["total_passenger_min_wait"],
                   s=100, label=f"{scen} {method}")
ax.set_xlabel("Fleet cost (λ × trips)")
ax.set_ylabel("Passenger-min chờ")
ax.set_title(f"Trade-off (λ={LAMBDA_COST}, {N_SLOTS} slots)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_tradeoff.png", dpi=120, bbox_inches="tight")
plt.show()

import time
lambda_rows = []
dem_peak = scenario_demand["weekday_peak"]
for lam in LAMBDA_GRID:
    t0 = time.time()
    lam_opt, sol_lam = resolve_lambda_for_demand_constrained(dem_peak, lam)
    m = evaluate_schedule(sol_lam, dem_peak, lambda_cost=lam)
    m_base = evaluate_schedule(baseline_trips, dem_peak, lambda_cost=lam)
    lambda_rows.append({
        "lambda_eval": lam,
        "lambda_opt": lam_opt,
        "total_trips": m["total_trips"],
        "total_passenger_min_wait": m["total_passenger_min_wait"],
        "total_fleet_cost": m["total_fleet_cost"],
        "objective": m["objective"],
        "baseline_objective": m_base["objective"],
        "improvement_pct": (m_base["objective"] - m["objective"]) / m_base["objective"] * 100,
        "pct_at_trips_max": float((sol_lam >= TRIPS_MAX).mean()) * 100,
        "elapsed_s": time.time() - t0,
    })
    lambda_sensitivity_df = pd.DataFrame(lambda_rows)
    print("=== Độ nhạy λ (weekday_peak, analytical) ===")
    print(lambda_sensitivity_df.round(2).to_string(index=False))
    lambda_sensitivity_df.to_csv(OUT_DIR / "lambda_sensitivity.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(lambda_sensitivity_df["lambda_eval"], lambda_sensitivity_df["total_trips"], "o-", label="Optimal")
    axes[0].axhline(baseline_trips.sum(), color="grey", ls="--", label="Baseline")
    axes[0].set_xlabel("λ"); axes[0].set_ylabel("Total trips"); axes[0].legend()
    axes[1].plot(lambda_sensitivity_df["lambda_eval"], lambda_sensitivity_df["objective"], "o-", label="Opt obj")
    axes[1].plot(lambda_sensitivity_df["lambda_eval"], lambda_sensitivity_df["baseline_objective"], "s--", label="Baseline")
    axes[1].set_xlabel("λ"); axes[1].set_ylabel("Objective"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_lambda_sensitivity.png", dpi=120, bbox_inches="tight")
    plt.show()

# --- Pareto frontier (Chebyshev scalarization) ---
def _pareto_optimize(demand, lambda_cost):
    return optimize_schedule_analytical(demand, lambda_cost=lambda_cost)

def _pareto_evaluate(trips, demand, lambda_cost):
    return evaluate_schedule(trips, demand, lambda_cost=lambda_cost)

pareto_df = srp.generate_pareto_frontier(
    scenario_demand,
    baseline_trips,
    optimize_fn=_pareto_optimize,
    evaluate_fn=_pareto_evaluate,
    scenario="weekday_peak",
    n_points=20,
    lambda_eval=float(LAMBDA_COST),
    lambda_scan=np.unique(np.round(np.geomspace(30, 2500, 60)).astype(float)),
)
knee_row = srp.find_knee_point(pareto_df)
print("=== Pareto frontier (weekday_peak, Chebyshev) ===")
print(pareto_df.round(2).to_string(index=False))
print(f"Knee: w={knee_row['w']:.2f} λ={knee_row['lambda_equiv']:.0f} "
      f"f1={knee_row['f1']:,.0f} f2={knee_row['f2']:,.0f} trips={knee_row['total_trips']:.0f}")
pareto_df.to_csv(OUT_DIR / "pareto_frontier.csv", index=False)

_base_peak = evaluate_schedule(baseline_trips, scenario_demand["weekday_peak"], lambda_cost=LAMBDA_COST)
fig, ax = plt.subplots(figsize=(9, 6))
srp.plot_pareto_frontier(
    pareto_df,
    baseline_f1=_base_peak["total_passenger_min_wait"],
    baseline_f2=_base_peak["total_fleet_cost"],
    ax=ax,
    title=f"Pareto frontier — weekday_peak (λ_eval={LAMBDA_COST}, {len(pareto_df)} pts)",
    save_path=OUT_DIR / "fig_pareto_frontier.png",
)
plt.show()
print("Đã lưu:", OUT_DIR / "pareto_frontier.csv", "|", OUT_DIR / "fig_pareto_frontier.png")


In [ ]:
def reshape_trip_matrix(sol: np.ndarray, direction: int, routes: list) -> pd.DataFrame:
    sub = pd.DataFrame({
        "route": slot_route, "dir": slot_dir, "hour": slot_hour, "trips": sol
    })
    sub = sub[sub["dir"] == direction]
    return sub.pivot(index="route", columns="hour", values="trips").reindex(routes)

HEATMAP_ROUTES = [r for r in route_totals.index.astype(str) if r in set(OPT_ROUTES)][:8]
if not HEATMAP_ROUTES:
    HEATMAP_ROUTES = list(OPT_ROUTES)[:8]
scen_to_show = "weekday_peak"
best_label = "Analytical" if USE_ANALYTICAL_OPT else "GA+Tabu"
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (method, sol) in zip(axes, [
    ("Baseline GTFS", baseline_trips),
    (best_label, opt_results[scen_to_show]),
]):
    m = reshape_trip_matrix(sol, direction=0, routes=HEATMAP_ROUTES)
    sns.heatmap(m, annot=False, fmt=".0f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"{method} — {scen_to_show} (dir 0, top-8 tuyến)")
    ax.set_xlabel("Hour"); ax.set_ylabel("Route")
plt.tight_layout()
plt.savefig(OUT_DIR / f"fig_schedule_{scen_to_show}.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
summary_metrics = {
    "run_experiment": str(RUN_EXPERIMENT),
    "demand_model_type": str(DEMAND_MODEL_TYPE),
    "n_slots": int(N_SLOTS),
    "n_routes": int(len(OPT_ROUTES)),
    "n_hours": int(len(OPT_HOURS)),
    "routes": list(OPT_ROUTES),
    "opt_method": OPT_METHOD,
    "n_headway_fallback": int(n_fallback),
    "scenarios": list(scenario_demand.keys()),
    "lambda_cost_default": float(LAMBDA_COST),
    "lambda_used": {k: float(v) for k, v in lambda_used.items()},
    "split_demand_by_direction": bool(SPLIT_DEMAND_BY_DIRECTION),
    "bound_status": bound_status,
    "bound_by_hour": bound_by_hour if "bound_by_hour" in dir() else {},
    "opt_target": str(globals().get("OPT_TARGET", "objective")),
    "balanced_weights": globals().get("BALANCED_WEIGHTS", {"wait": 0.5, "cost": 0.5}),
    "lambda_candidates": [float(x) for x in globals().get("LAMBDA_CANDIDATES", LAMBDA_GRID)],
    "baseline": {s: baseline_metrics[s] for s in scenario_demand},
    "optimized": {s: evaluate_schedule(opt_results[s], scenario_demand[s])
                  for s in scenario_demand},
    "nn_eval": nn_eval_metrics,
    "lambda_sensitivity_weekday_peak": lambda_sensitivity_df.to_dict(orient="records"),
    "pareto_frontier_weekday_peak": pareto_df.to_dict(orient="records"),
    "pareto_knee_weekday_peak": knee_row.to_dict(),
}
with open(OUT_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=2)
print("Đã ghi:", OUT_DIR / "baseline_metrics.json")


## 10. Kết luận & Discussion

### Tổng kết (29 tuyến, full run `default`)

- **Dữ liệu**: ridership chunked 7.29M (station×date×hour), 2023-12→2025-11; `factors_hourly`, GTFS `schedule_current/`; **1.182 slot** (giờ GTFS thực tế), 25 tuyến trong NN.
- **Đánh giá NN**: hold-out `autumn_2025` MAE **481** (−48.9%), R² **0.978**; 8-fold CV MAE **517±77** (−37.4% vs baseline). Xem `nn_eval_summary.csv`.
- **Tối ưu**: analytical + GTFS first/last; λ_opt=**260**; chờ giảm **~25%**, trips **+23%**, objective **~−21%** (λ_eval=150). `service_windows.csv`, `bound_by_hour.csv`.
- **GA/Tabu** (§7–8): có trong notebook, full run bỏ qua (N_SLOTS=1.182 > 200, analytical=True).

### Hạn chế (quan sát run hiện tại)

- Với bound uniform ×1.25, **~58% slot ở TRIPS_MAX**; dynamic bound theo giờ giảm ép trần peak, tăng headroom off-peak.
- Metric chờ = headway/2 theo slot; chưa mô hình transfer/walk.
- Chi phí = λ×trips (proxy), chưa ràng buộc crew/depot thực tế.
- 4/29 tuyến không có đủ ridership trong NN sau lọc giờ.
- Chưa so GA+Tabu trên full mạng (chỉ analytical khả thi).

### Hướng mở rộng

- Fine-tune `TRIPS_PEAK_MAX_FACTOR` / `TRIPS_OFFPEAK_MAX_FACTOR` hoặc tăng penalty bind để đạt interior > 60%.
- Báo cáo Pareto wait vs fleet thống nhất một λ.
- Ràng buộc fleet/depot; GTFS-RT / RL.
- Ablation 1 tuyến (+ GA/Tabu): `mta_schedule_optimization_single_route.ipynb`.
- So sánh config: `RUN_EXPERIMENT` → `OUT_DIR/` (`default` | `model_mlp` | `model_lstm`).
